In [42]:
pip -q install pymupdf

In [43]:
# 2) Configurare
import os, re, io, sys, json, time, math, bisect, unicodedata, collections, shutil, threading, warnings, functools
import datetime as dt
from pathlib import Path
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import requests
import geopandas as gpd
import shapely
from shapely.geometry import Point, LineString, Polygon, box, mapping, shape
from shapely.ops import transform as shp_transform
from pyproj import Transformer
from scipy.spatial import cKDTree
import pymupdf

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

PROIECT = Path("harta-sonde")
SITE = PROIECT / "docs"
DATA = SITE / "data"
BRUT = PROIECT / "date_brute"
for p in (SITE, DATA, BRUT):
    p.mkdir(parents=True, exist_ok=True)

PAGINA_ANRM = "https://www.namr.ro/inventar-sonde-etapa-1-conform-regulament-ue-2024-1787/"
PDF_SURSE = {
    "etapa1": dict(
        url="https://www.namr.ro/wp-content/uploads/2025/08/Inventarul-sondeloretapa-131.07.2025.pdf",
        titlu="Inventarul sondelor – Etapa I (publicat 5 august 2025)", cauta=["etapa-1", "etapa_1", "etapa1", "etapa 1", "31.07.2025", "sondeloretapa"]),
    "etapa2": dict(
        url="https://www.namr.ro/wp-content/uploads/2026/08/Inventar_sonde_ET_II_04.08.2026.pdf",
        titlu="Inventarul sondelor – Etapa II (publicat 5 august 2026)", cauta=["et_ii", "et ii", "etapa2", "etapa_2", "etapa 2", "04.08.2026"]),
}
TOTAL_ESTIMAT = 47135                      # estimarea ANRMPSG a sondelor vizate de regulament
RO_BBOX = (20.2, 43.5, 31.0, 48.3)         # lon_min, lat_min, lon_max, lat_max (include platforma Mării Negre)

ACTIV = dict(
    carbon_mapper=True, climate_trace=True, osm_sonde=True, osm_conducte=True,
    osm_sensibile=True, natura2000=True, eprtr=True, open_meteo=True, sentinel2=True,
)
CARBON_MAPPER_TOKEN = ""
DIST_SENSIBIL_M = 500
TIMEOUT = 90

ACCESAT = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
print("Pornit:", ACCESAT, "| Colab:", IN_COLAB)

Pornit: 2026-10-23 11:16 UTC | Colab: True


In [44]:
# 3) Registrul surselor de date + utilitare HTTP
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "harta-sonde-romania/1.0 (proiect open-source, educational)"})


def http(method, url, *, params=None, data=None, json_body=None, headers=None, timeout=TIMEOUT, retries=2, as_json=True):
    """Cerere HTTP cu reîncercări pentru erori temporare (429/5xx/timeout)."""
    last = None
    for i in range(retries + 1):
        try:
            r = SESSION.request(method, url, params=params, data=data, json=json_body, headers=headers, timeout=timeout)
        except requests.RequestException as e:
            last = e
            time.sleep(1.5 * (i + 1))
            continue
        if r.status_code in (429, 500, 502, 503, 504) and i < retries:
            last = requests.HTTPError(f"HTTP {r.status_code}")
            time.sleep(3 * (i + 1))
            continue
        r.raise_for_status()
        return r.json() if as_json else r
    raise last


SURSE = {}


def sursa(key, nume, furnizor, url, licenta, folosire, api=None):
    SURSE[key] = dict(key=key, nume=nume, furnizor=furnizor, url=url, licenta=licenta, folosire=folosire,
                      api=api, accesat=ACCESAT, status="neîncărcat", n=0, secunde=None)


def ruleaza(key, fn, *a, **kw):
    """Rulează un pas de încărcare; la eroare notează problema și continuă."""
    s = SURSE[key]
    if key in ACTIV and not ACTIV[key]:
        s["status"] = "dezactivat"
        print(f"  {s['nume']}: dezactivat")
        return None
    t = time.time()
    try:
        out = fn(*a, **kw)
        n = 0 if out is None else len(out)
        s.update(status="ok" if n else "fără înregistrări", n=int(n), secunde=round(time.time() - t, 1))
        print(f" {s['nume']}: {n:,} înregistrări în {time.time() - t:.1f}s")
        return out
    except Exception as e:
        s.update(status=f"eroare: {type(e).__name__}: {str(e)[:180]}", secunde=round(time.time() - t, 1))
        print(f"⚠️  {s['nume']}: {type(e).__name__}: {str(e)[:180]} — continui fără acest strat")
        return None


sursa("anrm", "Inventarul sondelor (Reg. UE 2024/1787)", "ANRMPSG – Autoritatea Națională de Reglementare în Domeniul Minier, Petrolier și al Stocării Geologice a CO₂",
      PAGINA_ANRM, "Informație publică (Legea 544/2001)",
      "Sondele inactive, închise temporar, închise permanent și abandonate: operator, status, coordonate Stereo70, emisii raportate.",
      api=PDF_SURSE["etapa2"]["url"])
sursa("geoboundaries", "Limitele județelor (ADM1)", "geoBoundaries – William & Mary geoLab",
      "https://www.geoboundaries.org/", "CC BY 4.0 (gbOpen)",
      "Verificarea coordonatelor (sonda cade în județul declarat?) și harta pe județe.",
      api="https://www.geoboundaries.org/api/current/gbOpen/ROU/ADM1/")
sursa("carbon_mapper", "Pene de metan detectate din satelit", "Carbon Mapper (EMIT/NASA, Tanager-1/Planet, AVIRIS-3 etc.)",
      "https://data.carbonmapper.org/", "Termenii de utilizare Carbon Mapper (data.carbonmapper.org)",
      "Pene de CH₄ cu debit estimat (kg/h), dată și instrument.",
      api="https://api.carbonmapper.org/api/v1/catalog/plumes/annotated")
sursa("climate_trace", "Surse de emisii din combustibili fosili", "Climate TRACE",
      "https://climatetrace.org/", "CC BY 4.0",
      "Câmpuri de petrol și gaze, mine de cărbune etc., cu emisii anuale estimate de CH₄ și CO₂e.",
      api="https://api.climatetrace.org/v6/assets")
sursa("osm_sonde", "Sonde cartate în OpenStreetMap", "Contribuitorii OpenStreetMap",
      "https://www.openstreetmap.org/copyright", "ODbL 1.0", "Obiecte man_made=petroleum_well.",
      api="https://overpass-api.de/api/interpreter")
sursa("osm_conducte", "Conducte de petrol și gaze (OSM)", "Contribuitorii OpenStreetMap",
      "https://www.openstreetmap.org/copyright", "ODbL 1.0", "Obiecte man_made=pipeline cu substance=gas/oil.",
      api="https://overpass-api.de/api/interpreter")
sursa("osm_sensibile", "Școli, grădinițe, spitale (OSM)", "Contribuitorii OpenStreetMap",
      "https://www.openstreetmap.org/copyright", "ODbL 1.0",
      f"Obiective sensibile; pe hartă apar doar cele la < {DIST_SENSIBIL_M} m de o sondă inventariată.",
      api="https://overpass-api.de/api/interpreter")
sursa("natura2000", "Arii protejate Natura 2000", "Agenția Europeană de Mediu (EEA) – Natura 2000 (versiunea 2024)",
      "https://www.eea.europa.eu/en/datahub/datahubitem-view/6fc8ad2d-195d-40f4-bdec-576e7d1268e4", "Reutilizare liberă cu atribuire (politica EEA)",
      "Situri SCI/SAC și SPA; câte sonde se află în interiorul lor.",
      api="https://bio.discomap.eea.europa.eu/arcgis/rest/services/ProtectedSites/Natura2000Sites/MapServer")
sursa("eprtr", "Facilități industriale raportoare (E-PRTR)", "Agenția Europeană de Mediu (EEA) – E-PRTR",
      "https://industry.eea.europa.eu/", "Reutilizare liberă cu atribuire (politica EEA)",
      "Facilități din extracția de petrol și gaze, rafinare și transport prin conducte.",
      api="https://air.discomap.eea.europa.eu/arcgis/rest/services/Air/EprtrFacilities_Dyna_WGS84/MapServer/0")
sursa("open_meteo", "Vânt în timp real", "Open-Meteo (modele ICON, GFS, ECMWF etc.)",
      "https://open-meteo.com/", "CC BY 4.0",
      "Viteza și direcția vântului la 10 m în zonele cu cea mai mare densitate de sonde, la momentul generării hărții.",
      api="https://api.open-meteo.com/v1/forecast")
sursa("sentinel2", "Imagini satelitare Sentinel-2 L2A", "Copernicus / ESA, prin Microsoft Planetary Computer",
      "https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a", "Date Copernicus Sentinel, acces liber și gratuit",
      "Cea mai clară scenă recentă peste zonele cu cele mai multe sonde (vizibilă la zoom ≥ 9).",
      api="https://planetarycomputer.microsoft.com/api/stac/v1/search")
sursa("carto", "Hartă de fundal", "Esri World Dark Gray Canvas (Esri, HERE, Garmin, © contribuitorii OpenStreetMap)",
      "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer",
      "Esri – termeni de utilizare, cu atribuire", "Fundal cartografic închis la culoare, fără cheie API.")
sursa("esri", "Fundal satelitar", "Esri World Imagery (Maxar, Earthstar Geographics)",
      "https://www.arcgis.com/home/item.html?id=10df2279f9684e4a9f6a7f08febac2a9", "Esri – termeni de utilizare", "Fundal satelitar opțional.")
print(f"{len(SURSE)} surse înregistrate")

13 surse înregistrate


In [45]:
def _etapa_din_nume(nume):
    n = nume.lower()
    for k, cfg in PDF_SURSE.items():
        if any(c in n for c in cfg["cauta"]):
            return k
    return None


PDF = {}
for d in (BRUT, Path("."), Path("/content")):
    if d.exists():
        for f in d.glob("*.pdf"):
            k = "etapa1" if f.stem == "etapa1" else "etapa2" if f.stem == "etapa2" else _etapa_din_nume(f.name)
            if k and k not in PDF and f.stat().st_size > 100_000:
                if f != BRUT / f"{k}.pdf":
                    shutil.copy(f, BRUT / f"{k}.pdf")
                PDF[k] = BRUT / f"{k}.pdf"

lipsa = [k for k in PDF_SURSE if k not in PDF]
if lipsa:
    if not IN_COLAB:
        raise FileNotFoundError(f"Puneți PDF-urile {lipsa} în folderul curent și rulați din nou.")
    from google.colab import files
    print("📤 Selectați PDF-urile (ambele odată):")
    for k in lipsa:
        print(f"   • {PDF_SURSE[k]['titlu']}  –  {PDF_SURSE[k]['url'].rsplit('/', 1)[-1]}")
    up = files.upload()
    for nume, continut in up.items():
        k = _etapa_din_nume(nume)
        if k is None and len(lipsa) == 1:
            k = lipsa[0]
        if k is None:
            print(f"   Nu recunosc „{nume}”: redenumiți-l etapa1.pdf sau etapa2.pdf și rulați din nou celula.")
            continue
        (BRUT / f"{k}.pdf").write_bytes(continut)
        PDF[k] = BRUT / f"{k}.pdf"
        print(f"    {nume} → {k}")

lipsa = [k for k in PDF_SURSE if k not in PDF]
if lipsa:
    raise FileNotFoundError(f"Lipsește PDF-ul pentru: {lipsa}. Rulați din nou celula și încărcați-l.")
print("PDF-uri pregătite:", {k: f"{v.stat().st_size / 1e6:.1f} MB" for k, v in PDF.items()})

PDF-uri pregătite: {'etapa2': '26.3 MB', 'etapa1': '4.8 MB'}


In [46]:
# 5) Parsarea rapidă a tabelelor din PDF
COLMAP = [  # (fragment din antetul PDF, nume canonic) – primul care se potrivește câștigă
    ("nr. crt", "nr_crt"), ("denumire operator", "operator"), ("adres", "adresa_operator"),
    ("încadrare", "status_raw"), ("județ", "judet_raw"), ("localitate", "localitate"),
    ("denumire sond", "denumire"), ("coordonata x", "x_raw"), ("coordonata y", "y_raw"), ("z (m)", "z_raw"),
    ("emisii", "metan_raw"), ("început", "data_inceput"), ("finalizare", "data_final"),
    ("orientare", "orientare"), ("adânci", "adancime_raw"), ("tipul sondei", "tip_sonda"),
    ("situarea", "situare"), ("număr aviz", "aviz_nr"), ("data aviz", "aviz_data"),
]


def antet_canonic(h):
    out = []
    for c in h:
        n = re.sub(r"\s+", " ", c.lower())
        out.append(next((v for k, v in COLMAP if k in n), re.sub(r"\W+", "_", n).strip("_")))
    return out


def _grila(page):
    V, H = [], []
    for d in page.get_drawings():
        for it in d["items"]:
            if it[0] == "re":
                r = it[1]
                if r.width < 2 and r.height > 3:
                    V.append((r.x0 + r.x1) / 2)
                elif r.height < 2 and r.width > 3:
                    H.append((r.y0 + r.y1) / 2)
            elif it[0] == "l":
                a, b = it[1], it[2]
                if abs(a.x - b.x) < 1 and abs(a.y - b.y) > 3:
                    V.append(a.x)
                elif abs(a.y - b.y) < 1 and abs(a.x - b.x) > 3:
                    H.append(a.y)

    def uneste(vals, tol=1.5):
        out = []
        for v in sorted(vals):
            if not out or v - out[-1] > tol:
                out.append(v)
        return out
    return uneste(V), uneste(H)


def parseaza_inventar(path):
    doc = pymupdf.open(path)
    antet, randuri = None, []
    for pno, page in enumerate(doc):
        V, H = _grila(page)
        if len(V) < 10 or len(H) < 2:
            continue
        ncol, nrow = len(V) - 1, len(H) - 1
        celule = collections.defaultdict(list)
        for x0, y0, x1, y1, w, b, l, n in page.get_text("words"):
            c = bisect.bisect(V, (x0 + x1) / 2) - 1
            r = bisect.bisect(H, (y0 + y1) / 2) - 1
            if 0 <= c < ncol and 0 <= r < nrow:
                celule[(r, c)].append((b, l, n, w))
        for r in range(nrow):
            vals = [" ".join(t[3] for t in sorted(celule.get((r, c), []))).strip() for c in range(ncol)]
            if not any(vals):
                continue
            if vals[0].lower().startswith("nr"):
                antet = antet_canonic(vals)
                continue
            if antet is None or len(vals) != len(antet):
                continue
            rec = dict(zip(antet, vals))
            rec["pagina"] = pno + 1
            randuri.append(rec)
    return pd.DataFrame(randuri)


t0 = time.time()
RAW = {k: parseaza_inventar(p) for k, p in PDF.items()}
for k, df in RAW.items():
    nr = pd.to_numeric(df["nr_crt"], errors="coerce")
    print(f"{k}: {len(df):,} rânduri, nr. crt. {int(nr.min())}–{int(nr.max())}, lipsă {int(nr.max() - nr.notna().sum())}, coloane: {len(df.columns)}")
print(f"Parsare: {time.time() - t0:.1f}s")
RAW["etapa2"].head(3)

etapa2: 20,781 rânduri, nr. crt. 1–20781, lipsă 0, coloane: 20
etapa1: 10,168 rânduri, nr. crt. 1–10168, lipsă 0, coloane: 19
Parsare: 34.5s


,nr_crt,operator,adresa_operator,status_raw,judet_raw,localitate,denumire,x_raw,y_raw,z_raw,metan_raw,data_inceput,data_final,orientare,adancime_raw,tip_sonda,situare,aviz_nr,aviz_data,pagina
0,1,SNGN ROMGAZ SA,"551130, Mediaş, P-ţa C.I. Motaş, nr.4",sonda inchisa permanent si abandonata,Alba,TIUR,1 TIUR,411403.160,518244.550,237.8,0,Fără informații,Fără informații,Verticală,267,Explorare,onshore,,,1
1,2,SNGN ROMGAZ SA,"551130, Mediaş, P-ţa C.I. Motaş, nr.4",sonda inchisa permanent si abandonata,Alba,CERGAU MIC,2 BROSTENI,418700.850,507562.820,469.0,0,6/13/1992,7/21/1992,Verticală,1690,Explorare,onshore,,,1
2,3,SNGN ROMGAZ SA,"551130, Mediaş, P-ţa C.I. Motaş, nr.4",sonda inchisa permanent si abandonata,Alba,CETATEA DE BALTA,121 CETATEA_DE_BALTA,438403.359,526042.764,356.8,0,4/26/1990,5/6/1990,Verticală,503,Exploatare,onshore,,,1


In [47]:
# 6) Curățare și normalizare (păstrăm și valorile originale din PDF pentru popup)
def norm_txt(s):
    s = str(s or "").replace("ã", "a").replace("Ã", "A")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).strip().upper()


def la_numar(s):
    s = str(s or "").replace(" ", "").replace(" ", "")
    if re.fullmatch(r"-?\d{1,3}(,\d{3})+(\.\d+)?", s):  # 342,156.94
        s = s.replace(",", "")
    s = s.replace(",", ".")
    try:
        v = float(s)
        return v if math.isfinite(v) else None
    except ValueError:
        return None


def an_din(s):
    s = str(s or "").strip()
    if re.fullmatch(r"\d{5}", s):  # dată serială Excel
        y = (dt.date(1899, 12, 30) + dt.timedelta(days=int(s))).year
        return y if 1850 < y < 2031 else None
    m = re.search(r"(18[5-9]\d|19\d\d|20[0-3]\d)", s)
    return int(m.group(1)) if m else None


STATUS_ETICHETE = {
    "inactiva": "Inactivă", "inchisa_temporar": "Închisă temporar",
    "inchisa_permanent": "Închisă permanent și abandonată", "abandonata": "Abandonată", "altul": "Altul",
}


def norm_status(s):
    n = norm_txt(s)
    if "PERMANENT" in n:
        return "inchisa_permanent"
    if "TEMPORAR" in n:
        return "inchisa_temporar"
    if "ABANDON" in n:
        return "abandonata"
    if "INACTIV" in n:
        return "inactiva"
    return "altul"


def curata(df, etapa):
    df = df.copy()
    for c in ["metan_raw", "aviz_nr", "aviz_data", "z_raw", "orientare", "tip_sonda", "situare", "data_final"]:
        if c not in df:
            df[c] = ""
    df["nr_crt"] = pd.to_numeric(df["nr_crt"], errors="coerce").astype("Int64")
    df["x"] = df["x_raw"].map(la_numar)
    df["y"] = df["y_raw"].map(la_numar)
    df["judet_key"] = df["judet_raw"].map(norm_txt)
    df["status"] = df["status_raw"].map(norm_status)
    df["metan_gh"] = df["metan_raw"].map(la_numar)
    df["an_foraj"] = df["data_inceput"].map(an_din).fillna(df["data_final"].map(an_din))
    df["adancime_m"] = df["adancime_raw"].map(la_numar)
    df["operator_key"] = df["operator"].map(lambda s: re.sub(r"[^A-Z0-9]", "", norm_txt(s)))
    df["etapa"] = etapa
    return df


E1 = curata(RAW["etapa1"], 1)
E2 = curata(RAW["etapa2"], 2)
print("Status (Etapa II):", E2["status"].value_counts().to_dict())
print("Valori emisii nenule:", int((E2["metan_gh"] > 0).sum()))

Status (Etapa II): {'inactiva': 8211, 'inchisa_permanent': 7615, 'inchisa_temporar': 4935, 'abandonata': 20}
Valori emisii nenule: 895


In [48]:
# 7) Limitele județelor (geoBoundaries)
def incarca_judete():
    url = None
    try:
        meta = http("GET", "https://www.geoboundaries.org/api/current/gbOpen/ROU/ADM1/")
        url = meta.get("simplifiedGeometryGeoJSON") or meta.get("gjDownloadURL")
        if meta.get("boundaryLicense"):
            SURSE["geoboundaries"]["licenta"] = meta["boundaryLicense"]
    except Exception as e:
        print("   API geoBoundaries indisponibil, folosesc copia de pe GitHub:", e)
    urls = [u for u in [url, "https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/main/releaseData/gbOpen/ROU/ADM1/geoBoundaries-ROU-ADM1_simplified.geojson"] if u]
    for u in urls:
        try:
            gj = http("GET", u, timeout=120)
            g = gpd.GeoDataFrame.from_features(gj["features"], crs=4326)
            g["judet_key"] = g["shapeName"].map(norm_txt)
            g["src_url"] = u
            return g[["shapeName", "shapeISO", "judet_key", "src_url", "geometry"]]
        except Exception as e:
            print("   eșec", u, e)
    raise RuntimeError("Nu am putut încărca limitele județelor")


JUDETE = ruleaza("geoboundaries", incarca_judete)
print(sorted(JUDETE["judet_key"]) if JUDETE is not None else "fără județe")

 Limitele județelor (ADM1): 42 înregistrări în 1.3s
['ALBA', 'ARAD', 'ARGES', 'BACAU', 'BIHOR', 'BISTRITA-NASAUD', 'BOTOSANI', 'BRAILA', 'BRASOV', 'BUCURESTI', 'BUZAU', 'CALARASI', 'CARAS-SEVERIN', 'CLUJ', 'CONSTANTA', 'COVASNA', 'DAMBOVITA', 'DOLJ', 'GALATI', 'GIURGIU', 'GORJ', 'HARGHITA', 'HUNEDOARA', 'IALOMITA', 'IASI', 'ILFOV', 'MARAMURES', 'MEHEDINTI', 'MURES', 'NEAMT', 'OLT', 'PRAHOVA', 'SALAJ', 'SATU MARE', 'SIBIU', 'SUCEAVA', 'TELEORMAN', 'TIMIS', 'TULCEA', 'VALCEA', 'VASLUI', 'VRANCEA']


In [49]:
# 8) Stereo70 → WGS84, cu detectarea coordonatelor X/Y inversate
T_S70 = Transformer.from_crs(3844, 4326, always_xy=True)   # intrare (Est, Nord)
T_WGS = Transformer.from_crs(4326, 3844, always_xy=True)
MAREA_NEAGRA = shp_transform(lambda x, y, z=None: T_WGS.transform(x, y), box(28.6, 43.4, 31.6, 45.4))


def rezolva_coordonate(df, judete):
    df = df.copy()
    ok = df["x"].between(100_000, 950_000) & df["y"].between(100_000, 950_000)
    jg = {} if judete is None else dict(zip(judete["judet_key"], judete.to_crs(3844).geometry))
    jg["MAREA NEAGRA"] = MAREA_NEAGRA
    geoms = np.array([jg.get(k) for k in df["judet_key"]], dtype=object)
    has = np.array([g is not None for g in geoms]) & ok.values
    x = df["x"].fillna(0).values
    y = df["y"].fillna(0).values
    dA = np.full(len(df), np.nan)
    dB = np.full(len(df), np.nan)
    if has.any():
        dA[has] = shapely.distance(geoms[has], shapely.points(y[has], x[has]))  # A: X=Nord (oficial)
        dB[has] = shapely.distance(geoms[has], shapely.points(x[has], y[has]))  # B: X=Est (inversat)
    alege = np.full(len(df), "", dtype=object)
    alege[has & (dA == 0) & (dB > 0)] = "A"
    alege[has & (dB == 0) & (dA > 0)] = "B"
    # orientarea majoritară pe operator (din rândurile neambigue)
    tmp = pd.DataFrame({"op": df["operator_key"], "a": alege})
    maj = tmp[tmp.a != ""].groupby("op")["a"].agg(lambda s: s.value_counts().idxmax()).to_dict()
    ambele = has & (dA == 0) & (dB == 0)
    niciuna = has & (dA > 0) & (dB > 0)
    fara_judet = ok.values & ~has
    for i in np.where(ambele | fara_judet)[0]:
        alege[i] = maj.get(df["operator_key"].iat[i], "A")
    alege[niciuna] = np.where(dA[niciuna] <= dB[niciuna], "A", "B")
    dist = np.where(alege == "A", dA, np.where(alege == "B", dB, np.nan))
    E = np.where(alege == "A", y, x)
    N = np.where(alege == "A", x, y)
    lon, lat = T_S70.transform(E, N)
    df["E"], df["N"], df["lon"], df["lat"] = E, N, lon, lat
    df["orientare_xy"] = np.select(
        [alege == "A", alege == "B"], ["X=Nord (convenția oficială)", "X=Est (X/Y inversate în PDF)"], "")
    df["metoda_xy"] = np.select([ambele, fara_judet, niciuna], ["majoritatea operatorului", "majoritatea operatorului", "cea mai apropiată de județ"], "verificat cu limita județului")
    df["dist_judet_km"] = np.round(dist / 1000, 2)
    df["verif"] = np.select(
        [~ok.values, fara_judet, dist == 0, dist <= 3000, dist <= 30000],
        ["exclus", "fara_judet", "ok", "margine", "suspect"], "exclus")
    df.loc[alege == "", "verif"] = "exclus"
    return df


E2 = rezolva_coordonate(E2, JUDETE)
E1 = rezolva_coordonate(E1, JUDETE)
print("Verificare coordonate (Etapa II):", E2["verif"].value_counts().to_dict())
print("Orientare:", E2["orientare_xy"].value_counts().to_dict())

# Ce sonde din Etapa II apar deja în Etapa I? (Etapa II este cumulativă)
def cheie(df):
    return df["denumire"].map(lambda s: re.sub(r"[^A-Z0-9]", "", norm_txt(s))) + "|" + df["E"].round(-1).astype(str) + "|" + df["N"].round(-1).astype(str)


E2["in_etapa1"] = cheie(E2).isin(set(cheie(E1)))
print(f"Sonde din Etapa II regăsite în Etapa I: {int(E2['in_etapa1'].sum()):,} din {len(E1):,}")
EXCLUSE = E2[E2["verif"] == "exclus"]
EXCLUSE.to_csv(BRUT / "sonde_excluse_coordonate.csv", index=False)
print(f"Sonde fără coordonate utilizabile (nu apar pe hartă, salvate în date_brute/): {len(EXCLUSE)}")

Verificare coordonate (Etapa II): {'ok': 20654, 'margine': 88, 'exclus': 36, 'suspect': 3}
Orientare: {'X=Nord (convenția oficială)': 17605, 'X=Est (X/Y inversate în PDF)': 3143, '': 33}
Sonde din Etapa II regăsite în Etapa I: 10,078 din 10,168
Sonde fără coordonate utilizabile (nu apar pe hartă, salvate în date_brute/): 36


In [50]:
# 9) GeoDataFrame-ul principal al sondelor
URL_PDF2 = PDF_SURSE["etapa2"]["url"]
S = E2[E2["verif"] != "exclus"].copy()
S = gpd.GeoDataFrame(S, geometry=gpd.points_from_xy(S["lon"], S["lat"]), crs=4326)
S["sid"] = S["nr_crt"].astype(int)
S["src_url"] = URL_PDF2 + "#page=" + S["pagina"].astype(str)
SURSE["anrm"].update(status="ok", n=len(S))
S_3844 = np.c_[S["E"].values, S["N"].values]
ARBORE_SONDE = cKDTree(S_3844)
print(f"{len(S):,} sonde pe hartă | {S['judet_key'].nunique()} județe | {S['operator'].nunique()} operatori")

20,745 sonde pe hartă | 35 județe | 16 operatori


In [51]:
# 10) Pene de metan — Carbon Mapper
def _g(d, *keys, default=None):
    """Caută o cheie în mai multe variante (API-urile își mai schimbă denumirile)."""
    if not isinstance(d, dict):
        return default
    low = {k.lower(): v for k, v in d.items()}
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
        if k.lower() in low and low[k.lower()] not in (None, ""):
            return low[k.lower()]
    return default


def _punct(geom):
    try:
        g = shape(geom) if isinstance(geom, dict) else geom
        return g if g.geom_type == "Point" else g.centroid
    except Exception:
        return None


def in_romania(lon, lat):
    return RO_BBOX[0] <= lon <= RO_BBOX[2] and RO_BBOX[1] <= lat <= RO_BBOX[3]


def carbon_mapper():
    base = "https://api.carbonmapper.org/api/v1/catalog/"
    headers = {"Authorization": f"Bearer {CARBON_MAPPER_TOKEN}"} if CARBON_MAPPER_TOKEN else None
    rez, limit, reusit, eroare = [], 1000, False, None
    for path in ("plumes/annotated", "plumes/related"):
        try:
            for page in range(10):
                params = [("bbox", v) for v in RO_BBOX] + [("plume_gas", "CH4"), ("limit", limit), ("offset", page * limit)]
                js = http("GET", base + path, params=params, headers=headers)
                items = js.get("items", js.get("results", js.get("features", []))) if isinstance(js, dict) else js
                rez += items
                if len(items) < limit:
                    break
            SURSE["carbon_mapper"]["api"] = base + path
            reusit = True
            break
        except Exception as e:
            print(f"   {path}: {e}")
            rez, eroare = [], e
    if not reusit:
        raise eroare
    rows = []
    for it in rez:
        props = it.get("properties", it) if isinstance(it, dict) else {}
        geom = _g(it, "geometry_json", "geometry", "point") or ({"type": "Point", "coordinates": [_g(props, "lon", "longitude"), _g(props, "lat", "latitude")]})
        p = _punct(geom)
        if p is None or not in_romania(p.x, p.y):
            continue
        pid = _g(props, "plume_id", "id", "name")
        gaz = str(_g(props, "gas", "plume_gas", default="CH4")).upper()
        if gaz != "CH4":
            continue
        rows.append(dict(
            pid=str(pid), em=la_numar(_g(props, "emission_auto", "emission_rate", "emission", "emissions")),
            em_unc=la_numar(_g(props, "emission_uncertainty_auto", "emission_uncertainty", "emission_rate_uncertainty")),
            data=str(_g(props, "scene_timestamp", "datetime", "acquisition_date", "published_at", default=""))[:10],
            platf=_g(props, "platform", default=""), instr=_g(props, "instrument", default=""),
            sector=_g(props, "sector", "ipcc_sector", default=""),
            img=_g(props, "plume_rgb_png", "plume_png", "plume_quicklook", "rgb_png", default=""),
            src_url=f"https://api.carbonmapper.org/api/v1/catalog/plume/{pid}",
            geometry=p))
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else gpd.GeoDataFrame(columns=["pid", "geometry"], geometry="geometry", crs=4326)


PENE = ruleaza("carbon_mapper", carbon_mapper)

 Pene de metan detectate din satelit: 48 înregistrări în 1.1s


In [52]:
# 11) Surse de emisii — Climate TRACE
def climate_trace():
    url = "https://api.climatetrace.org/v6/assets"
    rez, limit = [], 1000
    for page in range(10):
        js = http("GET", url, params={"countries": "ROU", "sectors": "fossil-fuel-operations", "limit": limit, "offset": page * limit})
        items = js.get("assets", js.get("items", [])) if isinstance(js, dict) else js
        rez += items or []
        if not items or len(items) < limit:
            break
    rows = []
    for a in rez:
        c = _g(a, "Centroid", "centroid") or {}
        coords = _g(c, "Geometry", "geometry", "coordinates")
        if isinstance(coords, dict):
            coords = coords.get("coordinates")
        if not coords or len(coords) < 2:
            continue
        lon, lat = float(coords[0]), float(coords[1])
        if not in_romania(lon, lat):
            continue
        ch4 = co2e = None
        for e in _g(a, "EmissionsSummary", "emissions_summary", "Emissions", default=[]) or []:
            gas = str(_g(e, "Gas", "gas", default="")).lower()
            q = la_numar(_g(e, "EmissionsQuantity", "emissions_quantity", "quantity"))
            if gas == "ch4":
                ch4 = q
            elif gas in ("co2e_100yr", "co2e-100", "co2e_100", "co2e"):
                co2e = q
        aid = _g(a, "Id", "id", "AssetId")
        rows.append(dict(
            aid=str(aid), nume=_g(a, "Name", "name", default=""),
            tip=_g(a, "AssetType", "Subsector", "subsector", "asset_type", default=""),
            sector=_g(a, "Sector", "sector", default=""), ch4_t=ch4, co2e_t=co2e,
            src_url=f"https://api.climatetrace.org/v6/assets/{aid}", geometry=Point(lon, lat)))
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else None


CT = ruleaza("climate_trace", climate_trace)

 Surse de emisii din combustibili fosili: 19 înregistrări în 0.2s


In [53]:
# 12) OpenStreetMap prin Overpass: sonde, conducte, obiective sensibile
OVERPASS = ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter",
            "https://maps.mail.ru/osm/tools/overpass/api/interpreter"]
ARIE_RO = 'area["ISO3166-1"="RO"][admin_level=2]->.ro;'


def overpass(q, key):
    last = None
    for u in OVERPASS:
        try:
            js = http("POST", u, data={"data": q}, timeout=200, retries=1)
            SURSE[key]["api"] = u
            return js.get("elements", [])
        except Exception as e:
            last = e
            print(f"   {u}: {e}")
    raise last


def _osm_pct(el):
    if "lat" in el:
        return el["lon"], el["lat"]
    c = el.get("center")
    return (c["lon"], c["lat"]) if c else (None, None)


def osm_sonde():
    els = overpass(f'[out:json][timeout:180];{ARIE_RO}nwr["man_made"="petroleum_well"](area.ro);out center tags;', "osm_sonde")
    rows = []
    for el in els:
        lon, lat = _osm_pct(el)
        if lon is None:
            continue
        t = el.get("tags", {})
        rows.append(dict(oid=f"{el['type']}/{el['id']}", nume=t.get("name") or t.get("ref") or "",
                         operator=t.get("operator", ""), subst=t.get("substance", ""),
                         src_url=f"https://www.openstreetmap.org/{el['type']}/{el['id']}", geometry=Point(lon, lat)))
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else None


def osm_conducte():
    els = overpass(f'[out:json][timeout:240];{ARIE_RO}way["man_made"="pipeline"]["substance"~"gas|oil|petroleum|fuel|hydrocarbons|condensate"](area.ro);out geom tags;', "osm_conducte")
    rows = []
    for el in els:
        g = el.get("geometry") or []
        if len(g) < 2:
            continue
        t = el.get("tags", {})
        line = LineString([(p["lon"], p["lat"]) for p in g]).simplify(0.0003)
        rows.append(dict(oid=f"way/{el['id']}", subst=t.get("substance", ""), operator=t.get("operator", ""),
                         nume=t.get("name", ""), loc=t.get("location", ""), diam=t.get("diameter", ""),
                         src_url=f"https://www.openstreetmap.org/way/{el['id']}", geometry=line))
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else None


TIP_SENSIBIL = {"school": "Școală", "kindergarten": "Grădiniță", "hospital": "Spital", "clinic": "Clinică"}


def osm_sensibile():
    els = overpass(f'[out:json][timeout:180];{ARIE_RO}nwr["amenity"~"^(school|kindergarten|hospital|clinic)$"](area.ro);out center tags;', "osm_sensibile")
    rows = []
    for el in els:
        lon, lat = _osm_pct(el)
        if lon is None:
            continue
        t = el.get("tags", {})
        rows.append(dict(oid=f"{el['type']}/{el['id']}", tip=TIP_SENSIBIL.get(t.get("amenity"), t.get("amenity")),
                         nume=t.get("name", ""), src_url=f"https://www.openstreetmap.org/{el['type']}/{el['id']}",
                         geometry=Point(lon, lat)))
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else None


OSM_SONDE = ruleaza("osm_sonde", osm_sonde)
OSM_CONDUCTE = ruleaza("osm_conducte", osm_conducte)
OSM_SENS = ruleaza("osm_sensibile", osm_sensibile)

 Sonde cartate în OpenStreetMap: 412 înregistrări în 38.0s
 Conducte de petrol și gaze (OSM): 279 înregistrări în 39.0s
 Școli, grădinițe, spitale (OSM): 8,281 înregistrări în 59.2s


In [54]:
# 13) Natura 2000 (EEA) și E-PRTR (EEA)
def arcgis_query(url, params, page_size=500, max_pages=40):
    feats = []
    for i in range(max_pages):
        p = dict(params, f="geojson", resultOffset=i * page_size, resultRecordCount=page_size)
        js = http("GET", url, params=p, timeout=180)
        if "error" in js:
            raise RuntimeError(js["error"])
        f = js.get("features", [])
        feats += f
        if len(f) < page_size and not js.get("exceededTransferLimit") and not js.get("properties", {}).get("exceededTransferLimit"):
            break
    return feats


def natura2000():
    base = "https://bio.discomap.eea.europa.eu/arcgis/rest/services/ProtectedSites/Natura2000Sites/MapServer"
    feats = []
    for layer in (0, 1, 2):  # SCI/SAC, SPA, ambele
        try:
            feats += arcgis_query(f"{base}/{layer}/query", dict(
                where="MS='RO'", outFields="SITECODE,SITENAME,SITETYPE,Area_km2", returnGeometry="true",
                outSR=4326, maxAllowableOffset=0.0015, geometryPrecision=5, orderByFields="OBJECTID"), page_size=200)
        except Exception as e:
            print(f"   Natura2000 strat {layer}: {e}")
    if not feats:
        raise RuntimeError("niciun sit returnat")
    g = gpd.GeoDataFrame.from_features(feats, crs=4326)
    g = g[g.geometry.notna()].copy()
    g["geometry"] = g.geometry.make_valid()
    g = g.dissolve(by="SITECODE", aggfunc="first").reset_index()
    g["src_url"] = "https://natura2000.eea.europa.eu/Natura2000/SDF.aspx?site=" + g["SITECODE"]
    return g


def eprtr():
    url = "https://air.discomap.eea.europa.eu/arcgis/rest/services/Air/EprtrFacilities_Dyna_WGS84/MapServer/0/query"
    out = "FacilityID,FacilityName,ParentCompanyName,City,ReportingYear,NACEActivityCode,IAActivityCode"
    where = ("CountryCode='RO' AND (NACEActivityCode LIKE '06%' OR NACEActivityCode LIKE '09.1%' OR "
             "NACEActivityCode LIKE '19%' OR NACEActivityCode LIKE '35.2%' OR NACEActivityCode LIKE '49.5%')")
    feats = arcgis_query(url, dict(where=where, outFields=out, outSR=4326, returnGeometry="true"), page_size=1000)
    if not feats:
        feats = arcgis_query(url, dict(where="CountryCode='RO'", outFields=out, outSR=4326, returnGeometry="true"), page_size=1000)
        feats = [f for f in feats if str(f["properties"].get("NACEActivityCode") or "").startswith(("06", "09.1", "19", "35.2", "49.5"))]
    g = gpd.GeoDataFrame.from_features(feats, crs=4326)
    g = g[g.geometry.notna()].sort_values("ReportingYear").drop_duplicates("FacilityID", keep="last")
    g["src_url"] = url + "?" + g["FacilityID"].map(lambda i: urlencode({"where": f"FacilityID={i}", "outFields": "*", "f": "html"}))
    return g


NATURA = ruleaza("natura2000", natura2000)
EPRTR = ruleaza("eprtr", eprtr)

 Arii protejate Natura 2000: 606 înregistrări în 18.2s
 Facilități industriale raportoare (E-PRTR): 27 înregistrări în 0.8s


In [55]:
# 14) Zonele cu cele mai multe sonde → vânt (Open-Meteo) și imagini Sentinel-2 (Planetary Computer)
def zone_fierbinti(n=12, celula=0.2, sep_km=35):
    c = S.assign(cx=(S.lon / celula).round().astype(int), cy=(S.lat / celula).round().astype(int))
    agg = c.groupby(["cx", "cy"]).agg(n=("sid", "size"), lon=("lon", "mean"), lat=("lat", "mean"),
                                      judet=("judet_raw", lambda s: s.mode().iat[0])).sort_values("n", ascending=False)
    ales = []
    for _, r in agg.iterrows():
        if all(math.dist((r.lon * 70, r.lat * 111), (a["lon"] * 70, a["lat"] * 111)) > sep_km for a in ales):
            ales.append(dict(lon=r.lon, lat=r.lat, n=int(r.n), judet=r.judet))
        if len(ales) >= n:
            break
    return ales


ZONE = zone_fierbinti()


def open_meteo():
    params = {"latitude": ",".join(f"{z['lat']:.4f}" for z in ZONE), "longitude": ",".join(f"{z['lon']:.4f}" for z in ZONE),
              "current": "wind_speed_10m,wind_direction_10m,wind_gusts_10m,temperature_2m",
              "wind_speed_unit": "kmh", "timezone": "Europe/Bucharest"}
    js = http("GET", "https://api.open-meteo.com/v1/forecast", params=params)
    js = js if isinstance(js, list) else [js]
    rows = []
    for z, r in zip(ZONE, js):
        cur = r.get("current", {})
        one = {"latitude": f"{z['lat']:.4f}", "longitude": f"{z['lon']:.4f}", "current": params["current"],
               "wind_speed_unit": "kmh", "timezone": "Europe/Bucharest"}
        rows.append(dict(viteza=cur.get("wind_speed_10m"), dir=cur.get("wind_direction_10m"), rafala=cur.get("wind_gusts_10m"),
                         temp=cur.get("temperature_2m"), ora=cur.get("time"), n_sonde=z["n"], judet=z["judet"],
                         src_url="https://api.open-meteo.com/v1/forecast?" + urlencode(one), geometry=Point(z["lon"], z["lat"])))
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=4326)


def sentinel2(n_zone=3, zile=120):
    stac = "https://planetarycomputer.microsoft.com/api/stac/v1"
    fin = dt.datetime.now(dt.timezone.utc)
    interval = f"{(fin - dt.timedelta(days=zile)).strftime('%Y-%m-%dT00:00:00Z')}/{fin.strftime('%Y-%m-%dT%H:%M:%SZ')}"
    rows, vazute = [], set()
    for z in ZONE[:n_zone]:
        js = http("POST", f"{stac}/search", json_body={
            "collections": ["sentinel-2-l2a"], "intersects": {"type": "Point", "coordinates": [z["lon"], z["lat"]]},
            "datetime": interval, "query": {"eo:cloud_cover": {"lt": 15}}, "limit": 50})
        items = sorted(js.get("features", []), key=lambda f: (round(f["properties"].get("eo:cloud_cover", 100) / 5), -pd.Timestamp(f["properties"]["datetime"]).value))
        if not items:
            continue
        it = items[0]
        if it["id"] in vazute:
            continue
        vazute.add(it["id"])
        q = {"collection": "sentinel-2-l2a", "item": it["id"], "assets": "visual", "asset_bidx": "visual|1,2,3", "nodata": 0}
        try:
            tj = http("GET", "https://planetarycomputer.microsoft.com/api/data/v1/item/tilejson.json", params=q)
            tiles = tj["tiles"][0]
        except Exception:
            tiles = "https://planetarycomputer.microsoft.com/api/data/v1/item/tiles/WebMercatorQuad/{z}/{x}/{y}@1x?" + urlencode(q)
        rows.append(dict(item=it["id"], data=it["properties"]["datetime"][:10],
                         nori=round(it["properties"].get("eo:cloud_cover", float("nan")), 1), tiles=tiles, judet=z["judet"],
                         src_url=f"{stac}/collections/sentinel-2-l2a/items/{it['id']}", geometry=shape(it["geometry"])))
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else None


VANT = ruleaza("open_meteo", open_meteo)
S2 = ruleaza("sentinel2", sentinel2)

 Vânt în timp real: 12 înregistrări în 0.5s
 Imagini satelitare Sentinel-2 L2A: 3 înregistrări în 2.4s


In [56]:
# 15) Analize spațiale (fiecare valoare derivată își păstrează sursa)
def in_3844(g):
    xs, ys = T_WGS.transform(g.geometry.x.values, g.geometry.y.values)
    return np.c_[xs, ys]


def cea_mai_apropiata_sonda(g):
    d, i = ARBORE_SONDE.query(in_3844(g))
    return d, S.iloc[i]


# a) sonde în arii Natura 2000
S["n2k"] = ""
S["n2k_nume"] = ""
if NATURA is not None:
    j = gpd.sjoin(S[["sid", "geometry"]], NATURA[["SITECODE", "SITENAME", "geometry"]], predicate="within", how="inner")
    agg = j.groupby(level=0).agg(n2k=("SITECODE", lambda s: ";".join(sorted(set(s)))), n2k_nume=("SITENAME", lambda s: "; ".join(sorted(set(s)))))
    S.loc[agg.index, "n2k"] = agg["n2k"]
    S.loc[agg.index, "n2k_nume"] = agg["n2k_nume"]
    NATURA["n_sonde"] = NATURA["SITECODE"].map(j["SITECODE"].value_counts()).fillna(0).astype(int)
    print(f"Sonde în Natura 2000: {int((S['n2k'] != '').sum()):,}")

# b) școli/spitale la < 500 m de o sondă
S["sens_n"] = 0
SENS_APROAPE = None
if OSM_SENS is not None and len(OSM_SENS):
    d, near = cea_mai_apropiata_sonda(OSM_SENS)
    OSM_SENS["d_m"] = d.round()
    OSM_SENS["sonda"] = near["denumire"].values
    OSM_SENS["sonda_sid"] = near["sid"].values
    OSM_SENS["sonda_st"] = near["status"].map(STATUS_ETICHETE).values
    arb = cKDTree(in_3844(OSM_SENS))
    vecini = arb.query_ball_point(S_3844, DIST_SENSIBIL_M)
    S["sens_n"] = [len(v) for v in vecini]
    dmin, imin = arb.query(S_3844)
    S["sens_d"] = np.where(dmin <= 2000, dmin.round(), np.nan)
    S["sens_nume"] = np.where(dmin <= 2000, (OSM_SENS["tip"] + " " + OSM_SENS["nume"].fillna("")).str.strip().values[imin], "")
    S["sens_oid"] = np.where(dmin <= 2000, OSM_SENS["oid"].values[imin], "")  # → openstreetmap.org/<oid>
    SENS_APROAPE = OSM_SENS[OSM_SENS["d_m"] <= DIST_SENSIBIL_M].copy()
    SENS_APROAPE["n_sonde_500"] = [len(ARBORE_SONDE.query_ball_point(p, DIST_SENSIBIL_M)) for p in in_3844(SENS_APROAPE)]
    print(f"Școli/grădinițe/spitale la < {DIST_SENSIBIL_M} m de o sondă: {len(SENS_APROAPE):,}")

# c) pene de metan ↔ cea mai apropiată sondă inventariată
if PENE is not None and len(PENE):
    d, near = cea_mai_apropiata_sonda(PENE)
    PENE["sonda_km"] = (d / 1000).round(2)
    PENE["sonda"] = near["denumire"].values
    PENE["sonda_sid"] = near["sid"].values
    PENE["sonda_op"] = near["operator"].values
    arb = cKDTree(in_3844(PENE))
    dp, ip = arb.query(S_3844)
    S["pana_km"] = np.where(dp <= 10_000, (dp / 1000).round(2), np.nan)
    S["pana_id"] = np.where(dp <= 10_000, PENE["pid"].values[ip], "")  # → API Carbon Mapper /catalog/plume/<id>

# d) Climate TRACE: câte sonde inventariate sunt la < 2 km
if CT is not None and len(CT):
    CT["n_sonde_2km"] = [len(v) for v in ARBORE_SONDE.query_ball_point(in_3844(CT), 2000)]

# e) sonde OSM care nu apar (încă) în inventar
if OSM_SONDE is not None and len(OSM_SONDE):
    d, near = cea_mai_apropiata_sonda(OSM_SONDE)
    OSM_SONDE["inv_m"] = d.round()
    OSM_SONDE["inv_sonda"] = near["denumire"].values
    print(f"Sonde OSM fără corespondent în inventar la < 150 m: {int((OSM_SONDE['inv_m'] > 150).sum()):,} din {len(OSM_SONDE):,}")

# f) statistici pe județe
JUD = None
if JUDETE is not None:
    st = S.groupby("judet_key").agg(
        n=("sid", "size"), inactiva=("status", lambda s: int((s == "inactiva").sum())),
        temporar=("status", lambda s: int((s == "inchisa_temporar").sum())),
        permanent=("status", lambda s: int(s.isin(["inchisa_permanent", "abandonata"]).sum())),
        ch4_n=("metan_gh", lambda s: int((s > 0).sum())), ch4_sum=("metan_gh", lambda s: float(s[s > 0].sum())),
        n2k=("n2k", lambda s: int((s != "").sum())), op=("operator", lambda s: s.value_counts().index[0]))
    JUD = JUDETE.merge(st, left_on="judet_key", right_index=True, how="left").fillna({"n": 0})
    JUD["km2"] = JUD.to_crs(3844).area / 1e6
    JUD["dens"] = (JUD["n"] / JUD["km2"] * 100).round(2)  # sonde / 100 km²

Sonde în Natura 2000: 974
Școli/grădinițe/spitale la < 500 m de o sondă: 155
Sonde OSM fără corespondent în inventar la < 150 m: 136 din 412


In [57]:
# 16) Export: GeoJSON + CSV + meta.json
def _curat(v):
    if v is None:
        return None
    if isinstance(v, (np.bool_, bool)):
        return bool(v)
    if isinstance(v, (np.integer, int)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return None if not math.isfinite(float(v)) else round(float(v), 4)
    if isinstance(v, (pd.Timestamp, dt.date)):
        return str(v)[:10]
    if v is pd.NA or v is pd.NaT:
        return None
    s = str(v)
    return s if s.strip() else None


def scrie_geojson(g, nume, coloane, precizie=1e-5):
    if g is None or len(g) == 0:
        return 0
    g = g.to_crs(4326)
    geoms = shapely.set_precision(np.array(g.geometry.values, dtype=object), precizie)
    feats = []
    for geom, rec in zip(geoms, g[coloane].to_dict("records")):
        if geom is None or geom.is_empty:
            continue
        props = {k: _curat(v) for k, v in rec.items()}
        feats.append({"type": "Feature", "geometry": mapping(geom), "properties": {k: v for k, v in props.items() if v is not None}})
    with open(DATA / f"{nume}.geojson", "w", encoding="utf-8") as f:
        json.dump({"type": "FeatureCollection", "features": feats}, f, ensure_ascii=False, separators=(",", ":"))
    print(f"   {nume}.geojson: {len(feats):,} obiecte, {(DATA / f'{nume}.geojson').stat().st_size / 1e6:.1f} MB")
    return len(feats)


SONDE_OUT = S.drop(columns=["x", "y"]).rename(columns={  # x/y = valorile exact ca în PDF
    "nr_crt": "nr", "operator": "op", "status": "st", "status_raw": "st_raw", "judet_raw": "jud", "localitate": "loc",
    "denumire": "den", "x_raw": "x", "y_raw": "y", "z_raw": "z", "metan_gh": "ch4", "metan_raw": "ch4_raw",
    "data_inceput": "dfi", "data_final": "dff", "orientare": "ori", "adancime_raw": "adm", "tip_sonda": "tip",
    "situare": "sit", "aviz_nr": "aviz", "aviz_data": "aviz_d", "pagina": "pg", "in_etapa1": "et1",
    "orientare_xy": "xy", "metoda_xy": "xy_m", "dist_judet_km": "dj", "an_foraj": "an"})
for c in ["sens_d", "sens_nume", "sens_oid", "pana_km", "pana_id"]:
    if c not in SONDE_OUT:
        SONDE_OUT[c] = None
inv = SONDE_OUT["xy"].str.startswith("X=Est")          # păstrăm nota doar unde X/Y au fost inversate
SONDE_OUT["xy"] = np.where(inv, "B", None)
SONDE_OUT["xy_m"] = np.where(inv, SONDE_OUT["xy_m"], None)
COL_SONDE = ["sid", "nr", "op", "st", "st_raw", "jud", "loc", "den", "x", "y", "z", "ch4", "ch4_raw", "dfi", "dff", "an",
             "ori", "adm", "tip", "sit", "aviz", "aviz_d", "pg", "et1", "xy", "xy_m", "verif", "dj", "n2k", "n2k_nume",
             "sens_n", "sens_d", "sens_nume", "sens_oid", "pana_km", "pana_id"]

for _f in DATA.glob("*.geojson"):
    _f.unlink()
print("Scriu fișierele:")
STRATURI = {}
STRATURI["sonde"] = scrie_geojson(SONDE_OUT, "sonde", COL_SONDE)
if JUD is not None:
    STRATURI["judete"] = scrie_geojson(JUD, "judete", ["shapeName", "shapeISO", "n", "inactiva", "temporar", "permanent", "ch4_n", "ch4_sum", "n2k", "op", "dens", "src_url"], 1e-4)
if NATURA is not None:
    STRATURI["natura2000"] = scrie_geojson(NATURA, "natura2000", ["SITECODE", "SITENAME", "SITETYPE", "Area_km2", "n_sonde", "src_url"], 1e-4)
if PENE is not None and len(PENE):
    STRATURI["carbon_mapper"] = scrie_geojson(PENE, "carbon_mapper", [c for c in ["pid", "em", "em_unc", "data", "platf", "instr", "sector", "img", "sonda", "sonda_sid", "sonda_km", "sonda_op", "src_url"] if c in PENE])
if CT is not None and len(CT):
    STRATURI["climate_trace"] = scrie_geojson(CT, "climate_trace", ["aid", "nume", "tip", "sector", "ch4_t", "co2e_t", "n_sonde_2km", "src_url"])
if OSM_SONDE is not None and len(OSM_SONDE):
    STRATURI["osm_sonde"] = scrie_geojson(OSM_SONDE, "osm_sonde", ["oid", "nume", "operator", "subst", "inv_m", "inv_sonda", "src_url"])
if OSM_CONDUCTE is not None and len(OSM_CONDUCTE):
    STRATURI["osm_conducte"] = scrie_geojson(OSM_CONDUCTE, "osm_conducte", ["oid", "subst", "operator", "nume", "loc", "diam", "src_url"])
if SENS_APROAPE is not None and len(SENS_APROAPE):
    STRATURI["osm_sensibile"] = scrie_geojson(SENS_APROAPE, "osm_sensibile", ["oid", "tip", "nume", "d_m", "sonda", "sonda_sid", "sonda_st", "n_sonde_500", "src_url"])
if EPRTR is not None and len(EPRTR):
    STRATURI["eprtr"] = scrie_geojson(EPRTR, "eprtr", ["FacilityID", "FacilityName", "ParentCompanyName", "City", "ReportingYear", "NACEActivityCode", "src_url"])
if VANT is not None and len(VANT):
    STRATURI["open_meteo"] = scrie_geojson(VANT, "vant", ["viteza", "dir", "rafala", "temp", "ora", "n_sonde", "judet", "src_url"])
if S2 is not None and len(S2):
    STRATURI["sentinel2"] = scrie_geojson(S2, "sentinel2", ["item", "data", "nori", "judet", "src_url"], 1e-4)

# CSV descărcabil cu inventarul curățat (coloanele originale + cele calculate)
col_csv = ["nr_crt", "operator", "adresa_operator", "status_raw", "status", "judet_raw", "localitate", "denumire", "x_raw", "y_raw",
           "z_raw", "metan_raw", "metan_gh", "data_inceput", "data_final", "an_foraj", "orientare", "adancime_raw", "tip_sonda",
           "situare", "aviz_nr", "aviz_data", "pagina", "in_etapa1", "orientare_xy", "verif", "lon", "lat", "n2k", "sens_n"]
S[[c for c in col_csv if c in S]].to_csv(DATA / "sonde_inventar_curatat.csv", index=False, encoding="utf-8-sig")

STAT = dict(
    total=int(len(E2)), pe_harta=int(len(S)), estimat=TOTAL_ESTIMAT, etapa1=int(len(E1)),
    procent=round(100 * len(E2) / TOTAL_ESTIMAT, 1), excluse=int(len(EXCLUSE)),
    status={k: int(v) for k, v in S["status"].value_counts().items()},
    ch4_raportat=int((S["metan_gh"] > 0).sum()), ch4_total_gh=round(float(S.loc[S["metan_gh"] > 0, "metan_gh"].sum()), 1),
    n2k=int((S["n2k"] != "").sum()), sens=int(len(SENS_APROAPE)) if SENS_APROAPE is not None else None,
    sonde_langa_sens=int((S["sens_n"] > 0).sum()),
    pene=int(len(PENE)) if PENE is not None else None,
    xy_inversat=int(S["orientare_xy"].str.startswith("X=Est").sum()), verif=S["verif"].value_counts().to_dict(),
    operatori=S["operator"].value_counts().head(8).to_dict(),
)
META = dict(
    titlu="Sondele uitate ale României", generat=ACCESAT, stat=STAT, surse=SURSE, straturi=STRATURI,
    status_etichete=STATUS_ETICHETE, pdf={k: v["url"] for k, v in PDF_SURSE.items()}, pagina_anrm=PAGINA_ANRM,
    operatori_adresa=S.groupby("operator")["adresa_operator"].first().to_dict(),
    sentinel=[] if S2 is None else [dict(r, bbox=[round(float(v), 5) for v in g.bounds]) for r, g in zip(S2[["item", "data", "nori", "tiles", "judet"]].to_dict("records"), S2.geometry)],
    fisiere=sorted(f.stem for f in DATA.glob("*.geojson")),
    bounds=[float(v) for v in S.total_bounds], dist_sensibil=DIST_SENSIBIL_M,
)
with open(DATA / "meta.json", "w", encoding="utf-8") as f:
    json.dump(META, f, ensure_ascii=False, indent=1, default=_curat)
print(json.dumps(STAT, ensure_ascii=False, indent=1))

Scriu fișierele:
   sonde.geojson: 20,745 obiecte, 9.7 MB
   judete.geojson: 42 obiecte, 0.5 MB
   natura2000.geojson: 606 obiecte, 1.7 MB
   carbon_mapper.geojson: 48 obiecte, 0.0 MB
   climate_trace.geojson: 19 obiecte, 0.0 MB
   osm_sonde.geojson: 412 obiecte, 0.1 MB
   osm_conducte.geojson: 279 obiecte, 0.1 MB
   osm_sensibile.geojson: 155 obiecte, 0.1 MB
   eprtr.geojson: 27 obiecte, 0.0 MB
   vant.geojson: 12 obiecte, 0.0 MB
   sentinel2.geojson: 3 obiecte, 0.0 MB
{
 "total": 20781,
 "pe_harta": 20745,
 "estimat": 47135,
 "etapa1": 10168,
 "procent": 44.1,
 "excluse": 36,
 "status": {
  "inactiva": 8200,
  "inchisa_permanent": 7590,
  "inchisa_temporar": 4935,
  "abandonata": 20
 },
 "ch4_raportat": 895,
 "ch4_total_gh": 24654.0,
 "n2k": 974,
 "sens": 155,
 "sonde_langa_sens": 341,
 "pene": 48,
 "xy_inversat": 3142,
 "verif": {
  "ok": 20654,
  "margine": 88,
  "suspect": 3
 },
 "operatori": {
  "OMV Petrom S.A.": 16813,
  "SNGN ROMGAZ SA": 1697,
  "DACIAN Petroleum S.A.": 1127,


In [58]:
# 17) Pagina web (MapLibre GL) — un singur index.html care citește data/*.geojson
INDEX_HTML = r"""<!doctype html>
<html lang="ro">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Sondele uitate ale României</title>
<meta name="description" content="Inventarul ANRMPSG al sondelor inactive, închise și abandonate (Reg. UE 2024/1787), lângă penele de metan detectate din satelit, ariile Natura 2000 și școlile din apropiere. Fiecare punct are sursa în popup.">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,600&family=IBM+Plex+Mono:wght@400;500&family=IBM+Plex+Sans:wght@400;500;600&display=swap" rel="stylesheet">
<link href="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css" rel="stylesheet">
<script src="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.js"></script>
<style>
:root{--bg:#0a0c0f;--panel:rgba(14,17,21,.94);--ink:#ebe8e2;--mute:#8d939c;--faint:#5b616a;--line:#252a31;
--acc:#22d3ee;--inactiva:#f2b134;--temporar:#f07a35;--permanent:#e0475f;--abandonata:#b06ef0;
--sans:"IBM Plex Sans",system-ui,sans-serif;--mono:"IBM Plex Mono",ui-monospace,monospace;--serif:"Fraunces",Georgia,serif}
*{box-sizing:border-box}html,body{margin:0;height:100%;background:var(--bg);color:var(--ink);font:14px/1.45 var(--sans)}
#map{position:fixed;inset:0}
a{color:var(--acc)}
#panel{position:fixed;z-index:2;top:12px;left:12px;bottom:12px;width:372px;background:var(--panel);backdrop-filter:blur(8px);
border:1px solid var(--line);border-radius:14px;display:flex;flex-direction:column;overflow:hidden}
#panel .scroll{overflow:auto;padding:20px 20px 8px;flex:1}
.kicker{font:500 11px/1 var(--mono);letter-spacing:.08em;text-transform:uppercase;color:var(--acc)}
h1{font:600 30px/1.05 var(--serif);margin:10px 0 10px;letter-spacing:-.01em}
.lede{color:#c4c1bb;margin:0 0 16px;font-size:13.5px}
h2{font:500 11px/1 var(--mono);letter-spacing:.08em;text-transform:uppercase;color:var(--mute);margin:20px 0 10px}
.big{display:flex;align-items:baseline;gap:10px;border-top:1px solid var(--line);padding-top:14px}
.big b{font:500 40px/1 var(--mono);letter-spacing:-.03em}
.big span{color:var(--mute);font-size:12.5px;line-height:1.3}
.bar{height:6px;background:#1c2026;border-radius:3px;margin:10px 0 4px;overflow:hidden;display:flex}
.bar i{display:block;height:100%}
.barlbl{display:flex;justify-content:space-between;font:11px var(--mono);color:var(--faint)}
.stats{display:grid;grid-template-columns:1fr 1fr;gap:1px;background:var(--line);border:1px solid var(--line);border-radius:10px;overflow:hidden;margin-top:14px}
.stat{background:#101318;padding:10px 12px}
.stat b{display:block;font:500 20px/1.1 var(--mono)}
.stat span{display:block;color:var(--mute);font-size:11.5px;line-height:1.3;margin-top:3px}
.stat a{font:10px var(--mono);color:var(--faint);text-decoration:none}
.stat a:hover{color:var(--acc)}
.status{margin-top:12px;font:12px var(--mono);color:var(--inactiva)}
.status[hidden]{display:none}
.search{position:relative;margin-top:14px}
.search input{width:100%;background:#0d1014;border:1px solid var(--line);color:var(--ink);border-radius:8px;padding:9px 11px;font:13px var(--sans)}
.search input:focus{outline:none;border-color:var(--acc)}
#qres{position:absolute;left:0;right:0;top:100%;background:#0f1216;border:1px solid var(--line);border-radius:8px;margin-top:4px;z-index:5;max-height:260px;overflow:auto}
#qres div{padding:7px 10px;cursor:pointer;border-bottom:1px solid #1a1e24;font-size:12.5px}
#qres div:hover{background:#161a20}#qres small{color:var(--mute)}
.layer{display:grid;grid-template-columns:18px 14px 1fr auto;align-items:center;gap:8px;padding:6px 0;border-bottom:1px solid #191d23;cursor:pointer}
.layer input{margin:0;accent-color:var(--acc)}
.sw{width:12px;height:12px;border-radius:50%}
.sw.sq{border-radius:2px}.sw.ln{height:3px;border-radius:2px}.sw.dm{transform:rotate(45deg);border-radius:1px;width:10px;height:10px}
.layer .t{font-size:13px;line-height:1.25}.layer .t small{display:block;color:var(--faint);font:10.5px var(--mono)}
.layer .n{font:11px var(--mono);color:var(--mute)}
.layer.off{opacity:.45;cursor:not-allowed}
.chips{display:flex;flex-wrap:wrap;gap:6px}
.chip{display:flex;align-items:center;gap:6px;border:1px solid var(--line);border-radius:99px;padding:4px 10px 4px 8px;font-size:12px;cursor:pointer;user-select:none}
.chip.dim{opacity:.35}.chip i{width:9px;height:9px;border-radius:50%}
.chk{display:flex;gap:8px;align-items:center;font-size:12.5px;color:#c4c1bb;margin-top:10px}
footer{border-top:1px solid var(--line);padding:10px 14px;display:flex;gap:8px;flex-wrap:wrap}
footer button,footer a{background:#151920;border:1px solid var(--line);color:var(--ink);border-radius:7px;padding:6px 10px;font:12px var(--sans);cursor:pointer;text-decoration:none}
footer button:hover,footer a:hover{border-color:var(--acc)}
.gen{font:10.5px var(--mono);color:var(--faint);padding:8px 20px;border-top:1px solid var(--line)}
@media (min-width:721px){.maplibregl-ctrl-bottom-right{max-width:calc(100% - 410px)}}
#toggle{display:none}
.maplibregl-popup{max-width:340px!important}
.maplibregl-popup-content{background:#0f1216;color:var(--ink);border:1px solid #2a3038;border-radius:10px;padding:14px 14px 10px;box-shadow:0 10px 30px rgba(0,0,0,.5);font:12.5px/1.4 var(--sans)}
.maplibregl-popup-tip{border-top-color:#2a3038!important;border-bottom-color:#2a3038!important}
.maplibregl-popup-close-button{color:var(--mute);font-size:18px;right:4px;top:2px}
.pp{max-height:min(62vh,560px);overflow:auto;padding-right:4px}
.pp-k{font:500 10.5px/1 var(--mono);letter-spacing:.06em;text-transform:uppercase;color:var(--mute);display:flex;gap:6px;align-items:center}
.pp-k i{width:8px;height:8px;border-radius:50%;display:inline-block}
.pp h3{font:600 17px/1.15 var(--serif);margin:6px 0 8px}
.pp table{border-collapse:collapse;width:100%}
.pp th{text-align:left;font-weight:400;color:var(--mute);padding:2px 8px 2px 0;vertical-align:top;white-space:nowrap;font-size:11.5px}
.pp td{padding:2px 0;font-variant-numeric:tabular-nums}
.pp .ctx{margin-top:8px;border-top:1px dashed #2a3038;padding-top:7px}
.pp .ctx div{margin:3px 0}
.pp sup a{font:9.5px var(--mono);text-decoration:none;color:var(--faint);border:1px solid #2a3038;border-radius:3px;padding:0 3px;margin-left:3px}
.pp sup a:hover{color:var(--acc);border-color:var(--acc)}
.pp .note{background:#15191f;border-left:2px solid var(--inactiva);padding:5px 8px;margin-top:8px;font-size:11.5px;color:#c4c1bb}
.pp .src{margin-top:10px;padding-top:8px;border-top:1px solid #2a3038;font-size:11.5px}
.pp .src b{font:500 10px var(--mono);letter-spacing:.06em;text-transform:uppercase;color:var(--acc);margin-right:4px}
.pp .src small{display:block;color:var(--faint);margin-top:2px;font-size:10.5px}
.pp img{width:100%;border-radius:6px;margin-top:8px;background:#000}
#modal{position:fixed;inset:0;z-index:10;background:rgba(0,0,0,.6);display:flex;align-items:center;justify-content:center;padding:16px}
#modal[hidden]{display:none}
#modal .box{background:#0f1216;border:1px solid var(--line);border-radius:14px;max-width:920px;width:100%;max-height:88vh;overflow:auto;padding:24px;position:relative}
#modal .x{position:absolute;right:12px;top:10px;background:none;border:0;color:var(--mute);font-size:24px;cursor:pointer}
#modal h2{margin-top:0}
#modal table{width:100%;border-collapse:collapse;font-size:12.5px}
#modal td,#modal th{border-bottom:1px solid var(--line);padding:8px 8px 8px 0;text-align:left;vertical-align:top}
#modal th{font:500 10.5px var(--mono);color:var(--mute);text-transform:uppercase;letter-spacing:.05em}
#modal li{margin:6px 0;color:#c9c6c0}
.okb{font:10.5px var(--mono);padding:1px 6px;border-radius:4px;background:#12301f;color:#5ee28f;white-space:nowrap}
.errb{font:10.5px var(--mono);padding:1px 6px;border-radius:4px;background:#3a1717;color:#ff8f8f}
@media (max-width:720px){
 #panel{top:auto;left:0;right:0;bottom:0;width:auto;max-height:62vh;border-radius:14px 14px 0 0;transition:transform .25s}
 #panel.closed{transform:translateY(calc(100% - 64px))}
 #toggle{display:block;position:absolute;top:0;left:0;right:0;height:22px;background:none;border:0;cursor:pointer}
 #toggle:before{content:"";display:block;width:40px;height:4px;border-radius:2px;background:#3a4048;margin:9px auto}
 h1{font-size:24px}.maplibregl-ctrl-top-right{top:0}
}
</style>
</head>
<body>
<div id="map"></div>
<aside id="panel">
 <button id="toggle" aria-label="Deschide panoul"></button>
 <div class="scroll">
  <div class="kicker">Reg. (UE) 2024/1787 · metan</div>
  <h1>Sondele uitate ale României</h1>
  <p class="lede">ANRMPSG publică etapizat, până în 2030, inventarul sondelor inactive, închise și abandonate. Harta pune inventarul lângă emisiile de metan observate din satelit și lângă ce se află în jurul sondelor. <b>Fiecare punct are sursa în popup.</b></p>
  <div id="hero"></div>
  <div id="status" class="status">⏳ Se încarcă cele peste 20.000 de sonde…</div>
  <div class="stats" id="stats"></div>
  <div class="search"><input id="q" placeholder="Caută o sondă, localitate sau operator…" autocomplete="off"><div id="qres" hidden></div></div>
  <h2>Straturi</h2><div id="layers"></div>
  <h2>Sonde după status</h2><div class="chips" id="chips"></div>
  <label class="chk"><input type="checkbox" id="onlych4"> doar sondele cu emisii de metan raportate &gt; 0</label>
 </div>
 <div class="gen" id="gen"></div>
 <footer><button data-m="surse">Surse de date</button><button data-m="metoda">Metodologie</button><a href="data/sonde_inventar_curatat.csv" download>⬇ CSV</a></footer>
</aside>
<div id="modal" hidden><div class="box"><button class="x" aria-label="Închide">×</button><div id="mbody"></div></div></div>
<script>
const STC={inactiva:'#f2b134',inchisa_temporar:'#f07a35',inchisa_permanent:'#e0475f',abandonata:'#b06ef0',altul:'#8d939c'};
const esc=s=>String(s??'').replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const fmt=(n,d=0)=>(n==null||n===''||isNaN(n))?'—':Number(n).toLocaleString('ro-RO',{maximumFractionDigits:d});
const fold=s=>String(s||'').normalize('NFD').replace(/[̀-ͯ]/g,'').replace(/ã/g,'a').toLowerCase();
let META,map,popup,DATA={},SID=new Map(),STATUS_ON=new Set(Object.keys(STC));
const EMPTY={type:'FeatureCollection',features:[]};
const SRC_ID={judete:'judete',natura2000:'natura',sentinel2:'s2',osm_conducte:'conducte',sonde:'sonde',osm_sonde:'osm-sonde',eprtr:'eprtr',climate_trace:'ct',osm_sensibile:'sens',carbon_mapper:'pene',vant:'vant'};

function src(key,url,extra){const s=META.surse[key]||{};return `<div class="src"><b>Sursa</b><a href="${esc(url||s.url)}" target="_blank" rel="noopener">${esc(s.nume)}</a>${extra?' · '+extra:''}<small>${esc(s.furnizor)} · ${esc(s.licenta)} · accesat ${esc(s.accesat)}</small></div>`}
function tag(key,url,label){const s=META.surse[key]||{};return `<sup><a href="${esc(url||s.url)}" target="_blank" rel="noopener" title="Sursa: ${esc(s.nume)} — ${esc(s.furnizor)}">${esc(label||key)}</a></sup>`}
function row(l,v,t=''){return (v==null||v==='')?'':`<tr><th>${l}</th><td>${typeof v==='number'?fmt(v,2):esc(v)}${t}</td></tr>`}
function wlink(sid,txt){return `<a href="#" data-sid="${sid}">${esc(txt)}</a>`}

const POP={
 sonde:p=>{const st=META.status_etichete[p.st]||p.st;const pdf=META.pdf.etapa2+'#page='+p.pg;
  let ctx='';
  if(p.n2k){ctx+=`<div>🌿 În aria Natura 2000 ${p.n2k.split(';').map(c=>`<b>${esc(c)}</b>`).join(', ')} – ${esc(p.n2k_nume)} ${tag('natura2000','https://natura2000.eea.europa.eu/Natura2000/SDF.aspx?site='+p.n2k.split(';')[0],'EEA')}</div>`}
  if(p.sens_d!=null){ctx+=`<div>🏫 ${p.sens_n?`<b>${p.sens_n}</b> școli/grădinițe/spitale la &lt; ${META.dist_sensibil} m · `:''}cel mai apropiat: ${esc(p.sens_nume||'obiectiv fără nume')}, <b>${fmt(p.sens_d)} m</b> ${tag('osm_sensibile',p.sens_oid?'https://www.openstreetmap.org/'+p.sens_oid:null,'OSM')}</div>`}
  if(p.pana_km!=null){ctx+=`<div>🛰️ Cea mai apropiată pană de metan detectată din satelit: <b>${fmt(p.pana_km,2)} km</b> ${tag('carbon_mapper',p.pana_id?'https://api.carbonmapper.org/api/v1/catalog/plume/'+encodeURIComponent(p.pana_id):null,'Carbon Mapper')}</div>`}
  ctx+=`<div>📄 Publicată în ${p.et1?'Etapa I (2025) și Etapa II (2026)':'Etapa II (2026)'} ${tag('anrm',META.pagina_anrm,'ANRMPSG')}</div>`;
  let note='';
  if(p.xy==='B')note+=`<div class="note">În PDF, coordonatele X/Y ale acestui rând sunt inversate față de convenția Stereo70 (X = Nord). Le-am interpretat ca X = Est (${esc(p.xy_m)}).</div>`;
  if(p.verif==='suspect'||p.verif==='margine')note+=`<div class="note">⚠ Punctul cade la ${fmt(p.dj,1)} km în afara județului declarat (${esc(p.jud)}). Coordonata din sursă poate fi eronată.</div>`;
  return `<div class="pp"><div class="pp-k"><i style="background:${STC[p.st]}"></i>Sondă · ${esc(st)}</div><h3>${esc(p.den)}</h3><table>
  ${row('Operator',p.op)}${row('Localitate',[p.loc,p.jud].filter(Boolean).join(', '))}${row('Încadrare',p.st_raw)}
  ${row('Emisii metan',p.ch4_raw!=null?p.ch4_raw+(p.ch4!=null?' g/h':''):'nespecificat')}
  ${row('Început foraj',p.dfi)}${row('Final foraj',p.dff)}${row('Adâncime',p.adm,p.adm?' m':'')}${row('Tip',p.tip)}${row('Orientare',p.ori)}${row('Situare',p.sit)}
  ${row('Aviz abandonare',[p.aviz,p.aviz_d].filter(Boolean).join(' din '))}${row('Stereo70 (PDF)',`X ${p.x} · Y ${p.y}`)}${row('Altitudine Z',p.z)}
  </table><div class="ctx">${ctx}</div>${note}
  ${src('anrm',pdf,`pagina ${p.pg}, nr. crt. ${p.nr} · <a href="${esc(META.pagina_anrm)}" target="_blank" rel="noopener">pagina oficială</a>`)}</div>`},
 pene:p=>`<div class="pp"><div class="pp-k"><i style="background:#22d3ee"></i>Pană de metan · satelit</div><h3>${p.em!=null?fmt(p.em)+' kg/h':'Pană CH₄'}${p.em_unc!=null?` <small style="font:12px var(--mono);color:var(--mute)">± ${fmt(p.em_unc)}</small>`:''}</h3><table>
  ${row('Data',p.data)}${row('Platformă',[p.platf,p.instr].filter(Boolean).join(' / '))}${row('Sector',p.sector)}${row('ID',p.pid)}</table>
  ${p.sonda?`<div class="ctx"><div>Cea mai apropiată sondă inventariată: ${wlink(p.sonda_sid,p.sonda)} (${esc(p.sonda_op||'')}), <b>${fmt(p.sonda_km,2)} km</b> ${tag('anrm',META.pagina_anrm,'ANRMPSG')}</div></div>`:''}
  ${p.img?`<img src="${esc(p.img)}" alt="Imaginea penei" loading="lazy" onerror="this.remove()">`:''}
  <div class="note" style="border-color:#22d3ee">Sateliții detectează de obicei doar scurgeri mari (zeci–sute de kg/h). Pana poate proveni și de la infrastructură activă, nu neapărat de la o sondă abandonată.</div>
  ${src('carbon_mapper',p.src_url,`<a href="https://data.carbonmapper.org/" target="_blank" rel="noopener">portal</a>`)}</div>`,
 ct:p=>`<div class="pp"><div class="pp-k"><i style="background:#a3e635"></i>Sursă de emisii · estimare</div><h3>${esc(p.nume||'Activ Climate TRACE')}</h3><table>
  ${row('Tip',p.tip)}${row('Sector',p.sector)}${row('CH₄ (an)',p.ch4_t!=null?fmt(p.ch4_t)+' t':null)}${row('CO₂e 100 ani',p.co2e_t!=null?fmt(p.co2e_t)+' t':null)}${row('Sonde inventariate la < 2 km',p.n_sonde_2km)}</table>
  ${src('climate_trace',p.src_url,`<a href="https://climatetrace.org/explore" target="_blank" rel="noopener">explorator</a>`)}</div>`,
 sens:p=>`<div class="pp"><div class="pp-k"><i style="background:#fff"></i>${esc(p.tip)}</div><h3>${esc(p.nume||p.tip+' fără nume în OSM')}</h3><table>
  ${row('Sonde la < '+META.dist_sensibil+' m',p.n_sonde_500)}</table><div class="ctx"><div>Cea mai apropiată sondă: ${wlink(p.sonda_sid,p.sonda)} – ${esc(p.sonda_st)}, <b>${fmt(p.d_m)} m</b> ${tag('anrm',META.pagina_anrm,'ANRMPSG')}</div></div>
  ${src('osm_sensibile',p.src_url,'element '+esc(p.oid))}</div>`,
 eprtr:p=>`<div class="pp"><div class="pp-k"><i style="background:#60a5fa"></i>Facilitate E-PRTR</div><h3>${esc(p.FacilityName)}</h3><table>
  ${row('Companie-mamă',p.ParentCompanyName)}${row('Localitate',p.City)}${row('Cod NACE',p.NACEActivityCode)}${row('Ultimul an raportat',p.ReportingYear)}${row('ID',p.FacilityID)}</table>
  ${src('eprtr',p.src_url)}</div>`,
 osm:p=>`<div class="pp"><div class="pp-k"><i style="background:#cbd5e1"></i>Sondă cartată în OSM</div><h3>${esc(p.nume||'Sondă fără nume')}</h3><table>
  ${row('Operator (OSM)',p.operator)}${row('Substanță',p.subst)}</table><div class="ctx"><div>${p.inv_m>150?'⚠ Nicio sondă inventariată la &lt; 150 m. Cea mai apropiată':'Corespunde probabil sondei inventariate'}: ${esc(p.inv_sonda)}, <b>${fmt(p.inv_m)} m</b> ${tag('anrm',META.pagina_anrm,'ANRMPSG')}</div></div>
  ${src('osm_sonde',p.src_url,'element '+esc(p.oid))}</div>`,
 conducte:p=>`<div class="pp"><div class="pp-k"><i style="background:#7dd3fc"></i>Conductă · ${esc(p.subst)}</div><h3>${esc(p.nume||p.operator||'Conductă')}</h3><table>
  ${row('Operator',p.operator)}${row('Amplasare',p.loc)}${row('Diametru',p.diam)}</table>${src('osm_conducte',p.src_url,'element '+esc(p.oid))}</div>`,
 vant:p=>`<div class="pp"><div class="pp-k"><i style="background:#fff"></i>Vânt · ${esc((p.ora||'').replace('T',' '))}</div><h3>${fmt(p.viteza,1)} km/h din ${fmt(p.dir)}°</h3><table>
  ${row('Rafale',p.rafala!=null?fmt(p.rafala,1)+' km/h':null)}${row('Temperatură',p.temp!=null?fmt(p.temp,1)+' °C':null)}${row('Zonă',p.judet)}${row('Sonde în celulă (~20 km)',p.n_sonde)}</table>
  <div class="note" style="border-color:#fff">Săgeata arată încotro ar fi purtat metanul la momentul generării hărții.</div>${src('open_meteo',p.src_url,'cerere API')}</div>`,
 s2:p=>`<div class="pp"><div class="pp-k"><i style="background:#e5e7eb"></i>Imagine Sentinel-2 L2A</div><h3>${esc(p.data)}</h3><table>${row('Acoperire nori',p.nori!=null?fmt(p.nori,1)+' %':null)}${row('Zonă',p.judet)}${row('Scenă',p.item)}</table>
  <div class="note" style="border-color:#e5e7eb">Măriți la zoom ≥ 9 ca să vedeți imaginea.</div>${src('sentinel2',p.src_url,'item STAC')}</div>`,
 natura:p=>`<div class="pp"><div class="pp-k"><i style="background:#3fb950"></i>Natura 2000 · ${({A:'SPA (păsări)',B:'SCI/SAC (habitate)',C:'SCI + SPA'})[p.SITETYPE]||esc(p.SITETYPE)}</div><h3>${esc(p.SITENAME)}</h3><table>
  ${row('Cod',p.SITECODE)}${row('Suprafață',p.Area_km2!=null?fmt(p.Area_km2,1)+' km²':null)}${row('Sonde inventariate în sit',p.n_sonde)}</table>${src('natura2000',p.src_url,'formularul standard al sitului')}</div>`,
 jud:p=>`<div class="pp"><div class="pp-k">Județ</div><h3>${esc(p.shapeName)}</h3><table>${row('Sonde inventariate',p.n)}${row('Inactive',p.inactiva)}${row('Închise temporar',p.temporar)}${row('Închise permanent / abandonate',p.permanent)}
  ${row('Densitate',p.dens!=null?fmt(p.dens,1)+' / 100 km²':null)}${row('Cu emisii raportate > 0',p.ch4_n)}${row('Total emisii raportate',p.ch4_sum?fmt(p.ch4_sum)+' g/h':null)}${row('În Natura 2000',p.n2k)}${row('Operator principal',p.op)}</table>
  <div class="ctx"><div>Statistici calculate din inventar ${tag('anrm',META.pagina_anrm,'ANRMPSG')}</div></div>${src('geoboundaries',p.src_url,'limita județului')}</div>`,
};

// ordinea de prioritate la click
const CLICK=[['vant','vant'],['pene','pene'],['sens','sens'],['ct','ct'],['eprtr','eprtr'],['sonde','sonde'],['osm-sonde','osm'],['conducte','conducte'],['s2-foot','s2'],['natura-fill','natura'],['jud-fill','jud']];

const GROUPS=[
 {k:'sonde',file:'sonde',layers:['sonde-heat','sonde'],label:'Sonde inventariate',src:'anrm',sw:'background:linear-gradient(90deg,#f2b134,#f07a35,#e0475f,#b06ef0)',on:true},
 {k:'pene',file:'carbon_mapper',layers:['pene-glow','pene'],label:'Pene de metan din satelit',src:'carbon_mapper',sw:'background:#22d3ee;box-shadow:0 0 8px #22d3ee',on:true},
 {k:'ct',file:'climate_trace',layers:['ct'],label:'Surse de emisii estimate',src:'climate_trace',sw:'background:#a3e635',cls:'dm',on:true},
 {k:'sens',file:'osm_sensibile',layers:['sens'],label:'Școli și spitale lângă sonde',src:'osm_sensibile',sw:'background:#fff;border:2px solid #60a5fa',on:true},
 {k:'natura',file:'natura2000',layers:['natura-fill','natura-line'],label:'Arii Natura 2000',src:'natura2000',sw:'background:rgba(63,185,80,.35);border:1px solid #3fb950',cls:'sq',on:true},
 {k:'vant',file:'vant',layers:['vant'],label:'Vântul acum',src:'open_meteo',sw:'background:#fff',cls:'dm',on:true},
 {k:'s2',file:'sentinel2',layers:['s2-foot'],label:'Imagini Sentinel-2 recente (zoom ≥ 9)',src:'sentinel2',sw:'background:#e5e7eb',cls:'sq',on:true},
 {k:'conducte',file:'osm_conducte',layers:['conducte'],label:'Conducte petrol și gaze',src:'osm_conducte',sw:'background:#7dd3fc',cls:'ln',on:false},
 {k:'osm',file:'osm_sonde',layers:['osm-sonde'],label:'Sonde cartate în OSM',src:'osm_sonde',sw:'background:#cbd5e1',on:false},
 {k:'eprtr',file:'eprtr',layers:['eprtr'],label:'Facilități industriale E-PRTR',src:'eprtr',sw:'background:#60a5fa',cls:'sq',on:false},
 {k:'jud',file:'judete',layers:['jud-fill','jud-line'],label:'Densitatea sondelor pe județe',src:'geoboundaries',sw:'background:linear-gradient(90deg,#2a2016,#f07a35)',cls:'sq',on:false},
 {k:'esri',file:null,layers:['esri'],label:'Fundal satelitar',src:'esri',sw:'background:#3b5d3a',cls:'sq',on:false},
];
const FILE_KEY={sonde:'anrm',carbon_mapper:'carbon_mapper',climate_trace:'climate_trace',osm_sensibile:'osm_sensibile',natura2000:'natura2000',vant:'open_meteo',sentinel2:'sentinel2',osm_conducte:'osm_conducte',osm_sonde:'osm_sonde',eprtr:'eprtr',judete:'geoboundaries'};

function icons(){
 const mk=(name,draw,s=48)=>{const c=document.createElement('canvas');c.width=c.height=s;const x=c.getContext('2d');x.fillStyle='#fff';draw(x,s);map.addImage(name,x.getImageData(0,0,s,s),{sdf:true,pixelRatio:2})};
 mk('sq',(x,s)=>x.fillRect(s*.2,s*.2,s*.6,s*.6));
 mk('dm',(x,s)=>{x.beginPath();x.moveTo(s/2,s*.1);x.lineTo(s*.9,s/2);x.lineTo(s/2,s*.9);x.lineTo(s*.1,s/2);x.fill()});
 mk('arrow',(x,s)=>{x.beginPath();x.moveTo(s/2,s*.04);x.lineTo(s*.82,s*.5);x.lineTo(s*.6,s*.5);x.lineTo(s*.6,s*.96);x.lineTo(s*.4,s*.96);x.lineTo(s*.4,s*.5);x.lineTo(s*.18,s*.5);x.fill()},64);
}

function addLayers(){
 const has=f=>(META.fisiere||[]).includes(f);
 const vis=k=>GROUPS.find(g=>g.k===k).on?'visible':'none';
 if(has('judete')){map.addSource('judete',{type:'geojson',data:EMPTY});
  map.addLayer({id:'jud-fill',type:'fill',source:'judete',layout:{visibility:vis('jud')},paint:{'fill-color':['interpolate',['linear'],['coalesce',['get','dens'],0],0,'rgba(0,0,0,0)',1,'#2a2016',5,'#6b3519',15,'#b24a22',40,'#f07a35'],'fill-opacity':.7}});
  map.addLayer({id:'jud-line',type:'line',source:'judete',layout:{visibility:vis('jud')},paint:{'line-color':'#3a3f47','line-width':.8}});}
 if(has('natura2000')){map.addSource('natura',{type:'geojson',data:EMPTY});
  map.addLayer({id:'natura-fill',type:'fill',source:'natura',layout:{visibility:vis('natura')},paint:{'fill-color':'#3fb950','fill-opacity':['interpolate',['linear'],['zoom'],6,.10,11,.16]}});
  map.addLayer({id:'natura-line',type:'line',source:'natura',layout:{visibility:vis('natura')},paint:{'line-color':'#3fb950','line-width':['interpolate',['linear'],['zoom'],6,.4,12,1.4],'line-opacity':.7}});}
 (META.sentinel||[]).forEach((s,i)=>{const b=s.bbox;
  map.addSource('s2r-'+i,{type:'raster',tiles:[s.tiles],tileSize:256,minzoom:8,maxzoom:14,...(b?{bounds:b}:{}),attribution:'Copernicus Sentinel-2 '+s.data.slice(0,4)+' / Planetary Computer'});
  map.addLayer({id:'s2r-'+i,type:'raster',source:'s2r-'+i,minzoom:8.5,layout:{visibility:vis('s2')},paint:{'raster-opacity':['interpolate',['linear'],['zoom'],8.5,0,9.5,.95]}});
  GROUPS.find(g=>g.k==='s2').layers.push('s2r-'+i);});
 if(has('sentinel2')){map.addSource('s2',{type:'geojson',data:EMPTY});
  map.addLayer({id:'s2-foot',type:'line',source:'s2',minzoom:7,layout:{visibility:vis('s2')},paint:{'line-color':'#e5e7eb','line-width':1,'line-dasharray':[3,3],'line-opacity':.5}});}
 if(has('osm_conducte')){map.addSource('conducte',{type:'geojson',data:EMPTY});
  map.addLayer({id:'conducte',type:'line',source:'conducte',layout:{visibility:vis('conducte'),'line-cap':'round'},paint:{'line-color':['case',['in','oil',['get','subst']],'#fbbf24','#7dd3fc'],'line-width':['interpolate',['linear'],['zoom'],6,.6,12,2],'line-opacity':.65}});}
 if(has('sonde')){map.addSource('sonde',{type:'geojson',data:EMPTY});
  map.addLayer({id:'sonde-heat',type:'heatmap',source:'sonde',maxzoom:9.5,layout:{visibility:vis('sonde')},paint:{'heatmap-radius':['interpolate',['linear'],['zoom'],5,4,9,14],'heatmap-intensity':['interpolate',['linear'],['zoom'],5,.5,9,1.4],
   'heatmap-color':['interpolate',['linear'],['heatmap-density'],0,'rgba(0,0,0,0)',.15,'#3b1d0f',.35,'#8a3418',.6,'#e0475f',.85,'#f2b134',1,'#fff3c4'],'heatmap-opacity':['interpolate',['linear'],['zoom'],7.5,.85,9.5,0]}});
  map.addLayer({id:'sonde',type:'circle',source:'sonde',layout:{visibility:vis('sonde')},paint:{'circle-color':['match',['get','st'],'inactiva',STC.inactiva,'inchisa_temporar',STC.inchisa_temporar,'inchisa_permanent',STC.inchisa_permanent,'abandonata',STC.abandonata,STC.altul],
   'circle-radius':['interpolate',['linear'],['zoom'],5,['case',['>',['coalesce',['get','ch4'],0],0],1.8,.9],9,['case',['>',['coalesce',['get','ch4'],0],0],4.5,2.6],14,['case',['>',['coalesce',['get','ch4'],0],0],9,6]],
   'circle-opacity':['interpolate',['linear'],['zoom'],5,.35,8,.9],'circle-stroke-color':'#22d3ee','circle-stroke-width':['interpolate',['linear'],['zoom'],7,0,10,['case',['>',['coalesce',['get','ch4'],0],0],1.2,0]]}});}
 if(has('osm_sonde')){map.addSource('osm-sonde',{type:'geojson',data:EMPTY});
  map.addLayer({id:'osm-sonde',type:'circle',source:'osm-sonde',layout:{visibility:vis('osm')},paint:{'circle-color':'#cbd5e1','circle-radius':['interpolate',['linear'],['zoom'],6,1.2,13,4],'circle-opacity':.7,'circle-stroke-color':['case',['>',['coalesce',['get','inv_m'],0],150],'#f2b134','rgba(0,0,0,0)'],'circle-stroke-width':1}});}
 if(has('eprtr')){map.addSource('eprtr',{type:'geojson',data:EMPTY});
  map.addLayer({id:'eprtr',type:'symbol',source:'eprtr',layout:{visibility:vis('eprtr'),'icon-image':'sq','icon-size':['interpolate',['linear'],['zoom'],5,.5,12,1],'icon-allow-overlap':true},paint:{'icon-color':'#60a5fa','icon-halo-color':'#0a0c0f','icon-halo-width':1}});}
 if(has('climate_trace')){map.addSource('ct',{type:'geojson',data:EMPTY});
  map.addLayer({id:'ct',type:'symbol',source:'ct',layout:{visibility:vis('ct'),'icon-image':'dm','icon-size':['interpolate',['linear'],['coalesce',['get','ch4_t'],0],0,.45,1000,.7,10000,1.1,50000,1.6],'icon-allow-overlap':true},paint:{'icon-color':'#a3e635','icon-halo-color':'#0a0c0f','icon-halo-width':1.5,'icon-opacity':.9}});}
 if(has('osm_sensibile')){map.addSource('sens',{type:'geojson',data:EMPTY});
  map.addLayer({id:'sens',type:'circle',source:'sens',layout:{visibility:vis('sens')},paint:{'circle-color':'#ffffff','circle-radius':['interpolate',['linear'],['zoom'],5,1.6,10,4,14,7],'circle-stroke-color':'#60a5fa','circle-stroke-width':['interpolate',['linear'],['zoom'],5,.6,10,2]}});}
 if(has('carbon_mapper')){map.addSource('pene',{type:'geojson',data:EMPTY});
  const r=['interpolate',['linear'],['coalesce',['get','em'],100],50,5,300,8,1000,12,5000,18,20000,26];
  map.addLayer({id:'pene-glow',type:'circle',source:'pene',layout:{visibility:vis('pene')},paint:{'circle-color':'#22d3ee','circle-radius':['*',2.2,r],'circle-blur':1,'circle-opacity':.35}});
  map.addLayer({id:'pene',type:'circle',source:'pene',layout:{visibility:vis('pene')},paint:{'circle-color':'rgba(34,211,238,.25)','circle-radius':r,'circle-stroke-color':'#22d3ee','circle-stroke-width':1.6}});}
 if(has('vant')){map.addSource('vant',{type:'geojson',data:EMPTY});
  map.addLayer({id:'vant',type:'symbol',source:'vant',layout:{visibility:vis('vant'),'icon-image':'arrow','icon-rotate':['+',['coalesce',['get','dir'],0],180],'icon-rotation-alignment':'map','icon-allow-overlap':true,
   'icon-size':['interpolate',['linear'],['coalesce',['get','viteza'],0],0,.45,10,.7,30,1.1]},paint:{'icon-color':'#ffffff','icon-opacity':.9,'icon-halo-color':'#0a0c0f','icon-halo-width':1.5}});}
}

function flyToSid(sid){const f=SID.get(+sid);if(!f)return;const c=f.geometry.coordinates;map.flyTo({center:c,zoom:Math.max(map.getZoom(),13.5),speed:1.6,offset:innerWidth>720?[190,-170]:[0,-200]});
 map.once('moveend',()=>{popup.setLngLat(c).setHTML(POP.sonde(f.properties)).addTo(map)});}

function filterSonde(){if(!map.getLayer('sonde'))return;const st=['in',['get','st'],['literal',[...STATUS_ON]]];
 const f=document.getElementById('onlych4').checked?['all',st,['>',['coalesce',['get','ch4'],0],0]]:st;map.setFilter('sonde',f);map.setFilter('sonde-heat',f);}

function ui(){
 const S=META.stat,sc=S.status||{};
 document.getElementById('hero').innerHTML=`<div class="big"><b>${fmt(S.total)}</b><span>sonde publicate până acum<br>din ~${fmt(S.estimat)} vizate de regulament</span></div>
 <div class="bar">${['inactiva','inchisa_temporar','inchisa_permanent','abandonata'].map(k=>`<i style="width:${100*(sc[k]||0)/S.estimat}%;background:${STC[k]}" title="${META.status_etichete[k]}: ${fmt(sc[k]||0)}"></i>`).join('')}</div>
 <div class="barlbl"><span>${S.procent}% inventariat</span><span>termen: 5 august 2030</span></div>`;
 const SHORT={anrm:'ANRMPSG',natura2000:'EEA',osm_sensibile:'OSM',carbon_mapper:'Carbon Mapper'};
 const st=(v,l,k)=>`<div class="stat"><b>${v==null?'—':fmt(v)}</b><span>${l}</span>${tag(k,null,SHORT[k])}</div>`;
 document.getElementById('stats').innerHTML=st(S.ch4_raportat,'sonde cu emisii de metan raportate > 0','anrm')+st(S.n2k,'sonde în interiorul unor arii Natura 2000','natura2000')
  +st(S.sens,`școli, grădinițe, spitale la < ${META.dist_sensibil} m de o sondă`,'osm_sensibile')+st(S.pene,'pene de metan detectate din satelit în România','carbon_mapper');
 document.querySelectorAll('.stat sup a').forEach(a=>a.style.marginLeft=0);
 const L=document.getElementById('layers');
 GROUPS.forEach(g=>{const ok=!g.file||(META.fisiere||[]).includes(g.file);const s=META.surse[g.src]||{};
  const n=g.file&&ok?'…':'';
  const d=document.createElement('label');d.className='layer'+(ok?'':' off');
  d.innerHTML=`<input type="checkbox" ${g.on&&ok?'checked':''} ${ok?'':'disabled'}><i class="sw ${g.cls||''}" style="${g.sw}"></i><span class="t">${g.label}<small>${ok?esc(s.furnizor||'').split(' – ')[0].split(' (')[0]:'indisponibil: '+esc((s.status||'').slice(0,60))}</small></span><span class="n">${n}</span>`;
  d.querySelector('input').onchange=e=>g.layers.forEach(id=>map.getLayer(id)&&map.setLayoutProperty(id,'visibility',e.target.checked?'visible':'none'));
  g.row=d;L.appendChild(d);});
 const C=document.getElementById('chips');
 ['inactiva','inchisa_temporar','inchisa_permanent','abandonata'].forEach(k=>{if(!sc[k])return;const c=document.createElement('span');c.className='chip';
  c.innerHTML=`<i style="background:${STC[k]}"></i>${META.status_etichete[k]} <small style="color:var(--mute);font-family:var(--mono)">${fmt(sc[k])}</small>`;
  c.onclick=()=>{STATUS_ON.has(k)?STATUS_ON.delete(k):STATUS_ON.add(k);c.classList.toggle('dim');filterSonde()};C.appendChild(c)});
 document.getElementById('onlych4').onchange=filterSonde;
 document.getElementById('gen').textContent='Generat '+META.generat+' · coordonate Stereo70 → WGS84';
 // căutare
 const q=document.getElementById('q'),qr=document.getElementById('qres');
 q.oninput=()=>{const t=fold(q.value.trim());if(t.length<2){qr.hidden=true;return}const out=[];
  if(!DATA.sonde){qr.innerHTML='<div><small>Sondele se încarcă încă…</small></div>';qr.hidden=false;return}
  for(const f of DATA.sonde.features){const p=f.properties;if(fold(p.den).includes(t)||fold(p.loc).includes(t)||fold(p.op).includes(t)){out.push(p);if(out.length>=12)break}}
  qr.innerHTML=out.length?out.map(p=>`<div data-sid="${p.sid}"><b>${esc(p.den)}</b> <small>${esc(p.loc)}, ${esc(p.jud)} · ${esc(META.status_etichete[p.st])}</small></div>`).join(''):'<div><small>Niciun rezultat</small></div>';qr.hidden=false};
 qr.onclick=e=>{const d=e.target.closest('[data-sid]');if(d){qr.hidden=true;flyToSid(d.dataset.sid);closePanelMobile()}};
 document.addEventListener('click',e=>{const a=e.target.closest('a[data-sid]');if(a){e.preventDefault();flyToSid(a.dataset.sid)}if(!e.target.closest('.search'))qr.hidden=true});
 // modale
 document.querySelectorAll('[data-m]').forEach(b=>b.onclick=()=>modal(b.dataset.m));
 document.querySelector('#modal .x').onclick=()=>document.getElementById('modal').hidden=true;
 document.getElementById('modal').onclick=e=>{if(e.target.id==='modal')e.target.hidden=true};
 document.getElementById('toggle').onclick=()=>document.getElementById('panel').classList.toggle('closed');
}
function closePanelMobile(){if(innerWidth<=720)document.getElementById('panel').classList.add('closed')}

function modal(which){const B=document.getElementById('mbody'),S=META.stat;
 if(which==='surse'){B.innerHTML=`<h2>Surse de date</h2><table><tr><th>Sursă</th><th>Folosită pentru</th><th>Licență</th><th>Stare</th></tr>${Object.values(META.surse).map(s=>`<tr>
  <td><a href="${esc(s.url)}" target="_blank" rel="noopener"><b>${esc(s.nume)}</b></a><br><small style="color:var(--mute)">${esc(s.furnizor)}</small>${s.api?`<br><small style="font-family:var(--mono);color:var(--faint);word-break:break-all">${esc(s.api)}</small>`:''}</td>
  <td>${esc(s.folosire)}</td><td>${esc(s.licenta)}</td><td>${s.status==='ok'?`<span class="okb">ok · ${fmt(s.n)}</span>`:(s.status==='neîncărcat'&&!s.api?'<span class="okb">fundal</span>':`<span class="errb">${esc(s.status)}</span>`)}<br><small style="color:var(--faint)">${esc(s.accesat)}</small></td></tr>`).join('')}</table>`}
 else{B.innerHTML=`<h2>Metodologie și limite</h2><ul>
 <li><b>Inventarul</b> provine din PDF-urile ANRMPSG (Etapa I, 31.07.2025 și Etapa II, 04.08.2026). Etapa II este cumulativă: ${fmt(S.total)} rânduri, din care cele ${fmt(S.etapa1)} din Etapa I. Tabelele au fost extrase automat din PDF, iar valorile originale sunt afișate neschimbate în popup.</li>
 <li><b>Coordonatele</b> din PDF sunt în Stereo70 (EPSG:3844) și au fost convertite în WGS84 cu pyproj. Convenția oficială este X = Nord, Y = Est, însă la ${fmt(S.xy_inversat)} sonde X și Y apar inversate (diferă de la un operator la altul). Pentru fiecare sondă am încercat ambele variante și am păstrat-o pe cea care cade în județul declarat. Când ambele variante cad în același județ, am folosit orientarea majoritară a operatorului.</li>
 <li>Verificare: ${Object.entries(S.verif||{}).map(([k,v])=>`${esc(k)}: ${fmt(v)}`).join(' · ')}. Sondele „margine” sau „suspect” au o notă în popup. ${fmt(S.excluse)} sonde nu au coordonate utilizabile (lipsesc sau sunt în alt sistem de coordonate) și nu apar pe hartă.</li>
 <li><b>Emisiile de metan</b> sunt valorile din coloana „Emisii metan (g/h)”, care există doar în Etapa II. Valorile goale, „-” și „N/A” sunt tratate ca lipsă. ${fmt(S.ch4_raportat)} sonde au o valoare &gt; 0.</li>
 <li><b>Penele detectate din satelit</b> (Carbon Mapper) apar doar peste un prag de detecție de ordinul zecilor sau sutelor de kg/h. O sondă abandonată emite de obicei mult mai puțin. Lipsa unei pene nu înseamnă lipsa emisiilor, iar o pană din apropiere nu e neapărat de la o sondă din inventar.</li>
 <li><b>Climate TRACE</b> publică estimări modelate, nu măsurători directe. <b>Vântul</b> (Open-Meteo) este cel de la momentul generării hărții.</li>
 <li><b>Distanțele</b> (școli/spitale la &lt; ${META.dist_sensibil} m, pene, sonde OSM) sunt calculate în metri, în Stereo70. Gradul de completitudine al OpenStreetMap variază de la o zonă la alta.</li>
 <li>Inventarul acoperă deocamdată ${S.procent}% din ~${fmt(S.estimat)} sonde. Lipsa sondelor dintr-o zonă poate însemna doar că nu au fost încă publicate.</li></ul>
 <p style="color:var(--mute);font-size:12px">Codul care generează harta este în același repository (notebook Python). Corecturile sunt binevenite.</p>`}
 document.getElementById('modal').hidden=false}

function setRow(file,txt,err){GROUPS.filter(g=>g.file===file).forEach(g=>{if(!g.row)return;g.row.querySelector('.n').textContent=txt;
 if(err){g.row.classList.add('off');g.row.querySelector('input').disabled=true;g.row.querySelector('small').textContent='eroare la încărcare: '+err}})}

async function main(){
 META=await (await fetch('data/meta.json',{cache:'no-cache'})).json();
 ui();
 const style={version:8,sources:{
  base:{type:'raster',tiles:['https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}'],tileSize:256,maxzoom:16,attribution:'Fundal: Esri, HERE, Garmin, © OpenStreetMap'},
  labels:{type:'raster',tiles:['https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Reference/MapServer/tile/{z}/{y}/{x}'],tileSize:256,maxzoom:16},
  esri:{type:'raster',tiles:['https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}'],tileSize:256,maxzoom:19,attribution:'Esri, Maxar, Earthstar Geographics'}},
  layers:[{id:'bg',type:'background',paint:{'background-color':'#0a0c0f'}},{id:'base',type:'raster',source:'base'},{id:'esri',type:'raster',source:'esri',layout:{visibility:'none'}}]};
 const wide=innerWidth>720;if(!wide)document.getElementById('panel').classList.add('closed');
 map=new maplibregl.Map({container:'map',style,bounds:[[20.2,43.55],[29.8,48.3]],fitBoundsOptions:{padding:wide?{left:400,right:40,top:40,bottom:40}:{left:10,right:10,top:10,bottom:200}},hash:true,attributionControl:false,maxZoom:17});
 map.addControl(new maplibregl.AttributionControl({compact:true,customAttribution:'Date: ANRMPSG, Carbon Mapper, Climate TRACE, © OpenStreetMap, EEA, Open-Meteo, Copernicus, geoBoundaries'}),'bottom-right');
 map.addControl(new maplibregl.NavigationControl({showCompass:false}),'top-right');
 map.addControl(new maplibregl.ScaleControl({unit:'metric'}),'bottom-right');
 popup=new maplibregl.Popup({maxWidth:'340px',closeButton:true});
 const mapReady=new Promise(res=>map.loaded()?res():map.on('load',res));
 const st=document.getElementById('status');
 // pornim toate descărcările imediat, în paralel cu încărcarea hărții
 const jobs=Object.keys(SRC_ID).filter(f=>(META.fisiere||[]).includes(f)).map(async f=>{
  try{const r=await fetch(`data/${f}.geojson`);if(!r.ok)throw new Error('HTTP '+r.status);const js=await r.json();
   await mapReady;DATA[f]=js;map.getSource(SRC_ID[f])?.setData(js);setRow(f,fmt(js.features.length));
   if(f==='sonde'){js.features.forEach(ft=>SID.set(ft.properties.sid,ft));st.hidden=true}}
  catch(e){setRow(f,'',e.message);if(f==='sonde')st.textContent='Sondele nu s-au putut încărca: '+e.message}});
 await mapReady;
 icons();addLayers();map.addLayer({id:'labels',type:'raster',source:'labels',paint:{'raster-opacity':.9}});filterSonde();
 map.on('click',e=>{const bb=[[e.point.x-4,e.point.y-4],[e.point.x+4,e.point.y+4]];
  for(const [id,pk] of CLICK){if(!map.getLayer(id)||map.getLayoutProperty(id,'visibility')==='none')continue;
   const fs=map.queryRenderedFeatures(bb,{layers:[id]});if(fs.length){const f=fs[0];
    const ll=f.geometry.type==='Point'?f.geometry.coordinates:e.lngLat;popup.setLngLat(ll).setHTML(POP[pk](f.properties)).addTo(map);return}}});
 CLICK.forEach(([id])=>{if(!map.getLayer(id))return;map.on('mouseenter',id,()=>map.getCanvas().style.cursor='pointer');map.on('mouseleave',id,()=>map.getCanvas().style.cursor='')});
 await Promise.all(jobs);
}
main().catch(e=>{document.getElementById('hero').innerHTML='<div class="note" style="color:#ff8f8f">Eroare la pornirea hărții: '+esc(e.message||e)+'</div>';console.error(e)});
</script>
</body>
</html>
"""
(SITE / "index.html").write_text(INDEX_HTML, encoding="utf-8")
(SITE / ".nojekyll").write_text("")


rows_surse = "\n".join(f"| [{s['nume']}]({s['url']}) | {s['furnizor']} | {s['licenta']} | {s['status']} |" for s in SURSE.values())
README = f"""# Sondele uitate ale României

Hartă interactivă a inventarului ANRMPSG al sondelor inactive, închise și abandonate (Regulamentul (UE) 2024/1787 privind reducerea emisiilor de metan), pusă lângă penele de metan detectate din satelit, ariile Natura 2000 și obiectivele sensibile din apropiere.

**{STAT['total']:,} sonde** publicate (Etapele I și II), din ~{TOTAL_ESTIMAT:,} vizate ({STAT['procent']}%). Pe hartă: {STAT['pe_harta']:,}.

Fiecare obiect de pe hartă are sursa în popup: pagina exactă din PDF-ul ANRMPSG, elementul OpenStreetMap, înregistrarea API etc.

## Surse de date

| Sursă | Furnizor | Licență | Stare la generare |
|---|---|---|---|
{rows_surse}

Generat la {ACCESAT} cu notebook-ul `harta_sonde_colab.ipynb`.

## Metodologie pe scurt
- Tabelele sunt extrase din PDF prin citirea grilei tabelului (PyMuPDF).
- Coordonatele Stereo70 (EPSG:3844) sunt convertite în WGS84. X/Y inversate au fost detectate la {STAT['xy_inversat']:,} sonde, prin verificare cu limita județului declarat.
- Detaliile și limitele sunt în pagina hărții, la „Metodologie și limite”.
"""
(PROIECT / "README.md").write_text(README, encoding="utf-8")
print("Site generat în", SITE.resolve())
for f in sorted(SITE.rglob("*")):
    if f.is_file():
        print(f"   {f.relative_to(PROIECT)}  {f.stat().st_size / 1e6:.2f} MB")

Site generat în /content/harta-sonde/docs
   docs/.nojekyll  0.00 MB
   docs/data/carbon_mapper.geojson  0.05 MB
   docs/data/climate_trace.geojson  0.01 MB
   docs/data/eprtr.geojson  0.01 MB
   docs/data/judete.geojson  0.52 MB
   docs/data/meta.json  0.01 MB
   docs/data/natura2000.geojson  1.73 MB
   docs/data/osm_conducte.geojson  0.10 MB
   docs/data/osm_sensibile.geojson  0.05 MB
   docs/data/osm_sonde.geojson  0.09 MB
   docs/data/sentinel2.geojson  0.00 MB
   docs/data/sonde.geojson  9.69 MB
   docs/data/sonde_inventar_curatat.csv  6.15 MB
   docs/data/vant.geojson  0.00 MB
   docs/index.html  0.04 MB


In [59]:
arhiva = shutil.make_archive("harta-sonde", "zip", PROIECT)
print("Arhivă:", arhiva)
if IN_COLAB:
    from google.colab import files
    files.download(arhiva)

Arhivă: /content/harta-sonde.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
import re
from pathlib import Path

TITLU = "Sonde abandonate"
AUTOR = "Ioana Birlan"

f = Path("harta-sonde/docs/index.html")
html = f.read_text(encoding="utf-8")


html = re.sub(r"<title>.*?</title>", f"<title>{TITLU}</title>", html, count=1)
html = re.sub(r"<h1>.*?</h1>", f"<h1>{TITLU}</h1>", html, count=1)


html = re.sub(r"<footer>.*?</footer>", f'<div class="autor">Hartă realizată de <b>{AUTOR}</b></div>', html, count=1, flags=re.S)
html = re.sub(r'<div class="autor">.*?</div>', f'<div class="autor">Hartă realizată de <b>{AUTOR}</b></div>', html, count=1)


html = html.replace("e.point.x-4,e.point.y-4],[e.point.x+4,e.point.y+4", "e.point.x-8,e.point.y-8],[e.point.x+8,e.point.y+8")


html = re.sub(r"<!--PERS-->.*?<!--/PERS-->", "", html, flags=re.S)
EXTRA = r"""<!--PERS-->
<style>
.autor{border-top:1px solid var(--line);padding:9px 20px;font:11px var(--mono);color:var(--faint)}
.autor b{color:var(--mute);font-weight:500}
.maplibregl-popup{max-width:400px!important}
.pp h4{font:500 10px var(--mono);letter-spacing:.07em;text-transform:uppercase;color:var(--acc);margin:10px 0 4px}
.pp table.full th{white-space:normal;width:46%;line-height:1.25}
.pp table.full td{word-break:break-word}
.pp table.full tr+tr th,.pp table.full tr+tr td{border-top:1px solid #1b2027}
.pp .gol{color:var(--faint)}
</style>
<script>
(function(){
  const vechi = POP.sonde;
  const v = x => (x==null || String(x).trim()==='') ? '<span class="gol">—</span>' : esc(x);
  POP.sonde = p => {
    const tmp = document.createElement('div'); tmp.innerHTML = vechi(p);
    const parte = sel => [...tmp.querySelectorAll(sel)].map(e => e.outerHTML).join('');
    const ft = SID.get(+p.sid); const c = ft ? ft.geometry.coordinates : null;
    const st = META.status_etichete[p.st] || p.st;
    const randuri = [
      ['Nr. crt.', p.nr],
      ['Denumire operator', p.op],
      ['Adresă operator', (META.operatori_adresa || {})[p.op]],
      ['Tip încadrare sondă conform Regulament', p.st_raw],
      ['Județ', p.jud],
      ['Localitate', p.loc],
      ['Denumire sondă', p.den],
      ['Coordonata X Stereo 70', p.x],
      ['Coordonata Y Stereo 70', p.y],
      ['Z (m)', p.z],
      ['Emisii metan (g/h)', p.ch4_raw],
      ['Dată început foraj', p.dfi],
      ['Dată finalizare foraj', p.dff],
      ['Orientarea sondei', p.ori],
      ['Adâncime sondă (m)', p.adm],
      ['Tipul sondei (explorare/exploatare)', p.tip],
      ['Situarea sondei', p.sit],
      ['Număr aviz de abandonare', p.aviz],
      ['Data aviz abandonare', p.aviz_d],
    ].map(([l, x]) => `<tr><th>${l}</th><td>${v(x)}</td></tr>`).join('');
    const gps = c ? `<tr><th>Coordonate WGS84</th><td>${c[1].toFixed(5)}, ${c[0].toFixed(5)} · <a href="https://www.openstreetmap.org/?mlat=${c[1]}&mlon=${c[0]}#map=16/${c[1]}/${c[0]}" target="_blank" rel="noopener">OSM</a> · <a href="https://www.google.com/maps?q=${c[1]},${c[0]}" target="_blank" rel="noopener">Google Maps</a></td></tr>` : '';
    return `<div class="pp"><div class="pp-k"><i style="background:${STC[p.st]}"></i>Sondă · ${esc(st)}</div><h3>${esc(p.den)}</h3>
      <h4>Date din inventarul ANRMPSG</h4><table class="full">${randuri}</table>
      <h4>Calculat</h4><table class="full">${gps}<tr><th>An foraj</th><td>${v(p.an)}</td></tr></table>
      <h4>Context</h4>${parte('.ctx')}${parte('.note')}${parte('.src')}</div>`;
  };
})();
</script>
<!--/PERS-->
"""
html = html.replace("</body>", EXTRA + "</body>", 1)
f.write_text(html, encoding="utf-8")

readme = Path("harta-sonde/README.md")
if readme.exists():
    rest = [l for l in readme.read_text(encoding="utf-8").split("\n")[1:] if not l.startswith("Hartă realizată de")]
    readme.write_text(f"# {TITLU}\n\nHartă realizată de {AUTOR}.\n" + "\n".join(rest), encoding="utf-8")

print("Pagina actualizată:", f)
import shutil
print("Arhivă nouă:", shutil.make_archive("harta-sonde", "zip", "harta-sonde"))
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, height=820)
except Exception:
    pass

Pagina actualizată: harta-sonde/docs/index.html
Arhivă nouă: /content/harta-sonde.zip


<IPython.core.display.Javascript object>